### **Cuaderno14 MCC225
#### **Evaluación experimental de sistemas multimodales con datos reales**

Este cuaderno es una base de trabajo individual. 


## Exposición 3

Exposición 3 contiene las siguientes carpetas
-data
-outputs
-scripts (Los scripts fueron utilizados para formar el dataset tanto para el entrenamiento (33 videos por 4 disciplinas, son 132 total) como para la prueba (fueron 10 videos por cada disciplina), siendo diferentes videos.

El repositorio contiene la carpeta notebook_MSR_VTT:
Cuaderno14_MSR-VTT.ipynb: La primera parte del experimento.
Cuaderno14_MSR-VTT33.ipynb: La segunda parte del experimento.
# Cuaderno14_MSR_VTT33.ipynb

Este cuaderno implementa el experimento multimodal sobre el subconjunto deportivo del conjunto de datos **MSR-VTT**. El objetivo es construir un pipeline que combine información visual y textual para la clasificación de videos deportivos y evaluar la robustez de modelos multimodales ante diferentes perturbaciones temporales.

## Objetivos

- Generar representaciones visuales mediante **CLIP** a partir de secuencias de frames.
- Generar descripciones automáticas de los frames utilizando **BLIP**.
- Construir un conjunto de datos multimodal que integre características visuales y textuales.
- Entrenar una capa de clasificación sobre los modelos **VisualBERT** y **LXMERT** mediante fine-tuning de la capa de salida.
- Evaluar el desempeño de los modelos utilizando un conjunto de prueba independiente.
- Analizar el efecto de perturbaciones temporales, como **Shuffle** y **Sampling**, sobre el rendimiento de los modelos.

## Flujo del experimento

1. Carga del conjunto de datos MSR-VTT Sports.
2. Extracción de embeddings visuales con CLIP.
3. Generación de captions mediante BLIP.
4. Construcción del dataset multimodal.
5. Conjunto de entrenamiento de 132 x 12 frames (procedentes de 132 videos).
6. Entrenar una capa de clasificación sobre los modelos **VisualBERT** y **LXMERT** mediante fine-tuning de la capa de salida.
7. Evaluación sobre el conjunto de prueba.
8. Evaluación con perturbaciones temporales y comparación de resultados.

## Modelos utilizados

- **CLIP (openai/clip-vit-base-patch32):** extracción de características visuales.
- **BLIP:** generación automática de descripciones de imágenes.
- **VisualBERT:** fusión multimodal para clasificación de deportes.
- **LXMERT:** arquitectura multimodal basada en atención cruzada para clasificación de deportes.

## Perturbaciones evaluadas

- **Original:** secuencia temporal completa.
- **Shuffle:** reordenamiento aleatorio de los frames.
- **Sampling:** reducción del número de frames manteniendo una muestra uniforme de la secuencia.

## Resultados esperados

Al finalizar este cuaderno se obtienen:

- Modelos VisualBERT y LXMERT entrenados.
- Métricas de evaluación (Accuracy, Precision, Recall y F1-score).
- Matrices de confusión.
- Comparación del rendimiento bajo diferentes perturbaciones temporales.

# Estructura del proyecto

La organización de archivos utilizada para el experimento es la siguiente:

```text
MCC225/
│
├── data/
│   │
│   ├── msr-vtt/
│   │   ├── frames/
│   │   │   └── context_12/
│   │   │
│   │   └── frames-test/
│   │       └── context_12/
│   │
│   └── processed/
│       └── multimodal_train_context_12.pt
│
├── outputs/
│   │
│   ├── datasets/
│   │
│   ├── embeddings/
│   │
│   ├── lxmert_dataset/
│   │
│   ├── models/
│   │
│   └── tables/
│
├── sports33_msr-vtt_train.csv
│
├── sports33_msr-vtt_test.csv

```

## Descripción de carpetas principales

- **data/msr-vtt/frames/**: contiene las secuencias de frames extraídas de los videos utilizados para entrenamiento. Cada muestra está representada por un conjunto de 12 frames.

- **data/msr-vtt/frames-test/**: contiene las secuencias de frames correspondientes al conjunto de prueba.

- **data/processed/**: almacena los datasets procesados en formato PyTorch, incluyendo las representaciones multimodales.

- **outputs/embeddings/**: contiene los embeddings visuales generados durante la extracción de características.

- **outputs/datasets/**: almacena los datasets multimodales preparados para entrenamiento y evaluación.

- **outputs/lxmert_dataset/**: contiene los archivos adaptados al formato requerido por LXMERT.

- **outputs/models/**: almacena los modelos entrenados y pesos obtenidos durante los experimentos.

- **outputs/tables/**: contiene tablas generadas durante el procesamiento y análisis de resultados.

- **sports33_msr-vtt_train.csv** y **sports33_msr-vtt_test.csv**: archivos de metadatos con la información de las muestras utilizadas para entrenamiento y prueba.


#### **Dependencias**

Se reutiliza el cuaderno14 del curso en los requerimientos.
En los scripts
Para descargar videos se utiliza:
pip install yt-dlp
Para cortar en frames se utiliza:
#pip install opencv-python-headless

In [ ]:
# %pip install -q torch torchvision transformers datasets pillow pandas numpy scikit-learn matplotlib tqdm nltk evaluate accelerate

### **Configuración reproducible**

#### **Semillas, rutas y parámetros**

Esta sección fija los parámetros del experimento. Cambiar estos valores debe quedar justificado en el reporte.

In [20]:
import json
import random
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import torch


# =====================================================
# REPRODUCIBILIDAD
# =====================================================

SEED = 22514

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# =====================================================
# CONFIGURACIÓN DEL EXPERIMENTO
# =====================================================

@dataclass
class ExperimentConfig:

    ####################################################
    # DATASETS
    ####################################################
    split: str = "train"
    
    train_metadata: str = "outputs/sports33_msr-vtt_train.csv"

    test_metadata: str = "outputs/sports33_msr-vtt_test.csv"


    ####################################################
    # MODELOS
    ####################################################

    clip_model_name: str = "openai/clip-vit-base-patch32"
    blip_model_name: str = "Salesforce/blip-image-captioning-base"
    use_blip_captioner: bool = True

    visualbert_model: str = "uclanlp/visualbert-vqa-coco-pre"

    lxmert_model: str = "unc-nlp/lxmert-base-uncased"

    ####################################################
    # FRAME DATASET
    ####################################################

    frame_context: str = "context_12"

    frame_root: str = "data/msr-vtt/frames"

    frame_metadata: str = (
        "outputs/embeddings/sports33_train_context_12_frame_metadata.csv"
    )
    ####################################################
    # TAREA
    ####################################################

    task: str = "text_to_frame_retrieval"


    ####################################################
    # RANKING TEXTO ↔ FRAME
    ####################################################

    ranking_strategy: str = "clip_similarity"

    captions_per_video: int = 20

    top_k_frames: int = 8


    ####################################################
    # FRAMES
    ####################################################

    frames_per_video: int = 12

    frame_strategy: str = "top8_clip"

    frame_strategies: tuple = (

        "central",

        "top4_clip",

        "top8_clip",

        "top16_clip",

        "average_embeddings"

    )


    ####################################################
    # PERTURBACIONES
    ####################################################

    perturbation: str = "none"

    perturbations: tuple = (

        "none",

        "reverse",

        "shuffle",

        "drop",

        "sampling"

    )


    ####################################################
    # TASA DE MUESTREO
    ####################################################

    sampling_rates: tuple = (

        1,

        2,

        4

    )


    ####################################################
    # EVALUACIÓN
    ####################################################

    batch_size: int = 16

    confidence_level: float = 0.95


    ####################################################
    # REPRODUCIBILIDAD
    ####################################################

    seed: int = SEED


    ####################################################
    # RUTAS
    ####################################################

    repo_root: str = "."


CONFIG = ExperimentConfig()


# =====================================================
# RUTAS
# =====================================================

ROOT = Path(CONFIG.repo_root).resolve()

DATA = ROOT / "data"

DATA_RAW = DATA / "raw"

DATA_VIDEOS = DATA / "videos"

DATA_FRAMES = DATA / "frames"

DATA_PROCESSED = DATA / "processed"


OUTPUTS = ROOT / "outputs"

OUTPUT_EMBEDDINGS = OUTPUTS / "embeddings"

OUTPUT_METRICS = OUTPUTS / "metrics"

OUTPUT_TABLES = OUTPUTS / "tables"

OUTPUT_FIGURES = OUTPUTS / "figures"


REPORTS = ROOT / "reports"


for folder in [

    DATA,

    DATA_RAW,

    DATA_VIDEOS,

    DATA_FRAMES,

    DATA_PROCESSED,

    OUTPUTS,

    OUTPUT_EMBEDDINGS,

    OUTPUT_METRICS,

    OUTPUT_TABLES,

    OUTPUT_FIGURES,

    REPORTS

]:

    folder.mkdir(
        parents=True,
        exist_ok=True
    )


print(f"\nDispositivo: {DEVICE}\n")

print(
    json.dumps(
        asdict(CONFIG),
        indent=2,
        ensure_ascii=False
    )
)


Dispositivo: cpu

{
  "split": "train",
  "train_metadata": "outputs/sports33_msr-vtt_train.csv",
  "test_metadata": "outputs/sports33_msr-vtt_test.csv",
  "clip_model_name": "openai/clip-vit-base-patch32",
  "blip_model_name": "Salesforce/blip-image-captioning-base",
  "use_blip_captioner": true,
  "visualbert_model": "uclanlp/visualbert-vqa-coco-pre",
  "lxmert_model": "unc-nlp/lxmert-base-uncased",
  "frame_context": "context_12",
  "frame_root": "data/msr-vtt/frames",
  "frame_metadata": "outputs/embeddings/sports33_train_context_12_frame_metadata.csv",
  "task": "text_to_frame_retrieval",
  "ranking_strategy": "clip_similarity",
  "captions_per_video": 20,
  "top_k_frames": 8,
  "frames_per_video": 12,
  "frame_strategy": "top8_clip",
  "frame_strategies": [
    "central",
    "top4_clip",
    "top8_clip",
    "top16_clip",
    "average_embeddings"
  ],
  "perturbation": "none",
  "perturbations": [
    "none",
    "reverse",
    "shuffle",
    "drop",
    "sampling"
  ],
  "samp

In [25]:
def split_captions(captions, simbolo_separacion="|||"):
    """
    Separa captions usando un símbolo separador configurable.

    Args:
        captions (str): texto con múltiples captions unidos.
        simbolo_separacion (str): símbolo utilizado para separar captions.

    Returns:
        list: lista de captions individuales limpios.
    """

    if captions is None:
        return []

    captions_list = captions.split(simbolo_separacion)

    captions_list = [
        caption.strip()
        for caption in captions_list
        if caption.strip()
    ]

    return captions_list

In [26]:
# ============================================
# Función: construir embeddings de captions
# ============================================

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm


def build_caption_embeddings(
    sample,
    model,
    processor,
    simbolo_separacion="|||"
):
    """
    Genera embeddings de captions usando CLIP.

    Parámetros
    ----------
    sample : str
        "train" o "test"
    """

    # ----------------------------------------
    # Seleccionar metadata
    # ----------------------------------------

    if sample == "train":

        metadata_file = ROOT / CONFIG.train_metadata
        output_name = "sports33_train"

    elif sample == "test":

        metadata_file = ROOT / CONFIG.test_metadata
        output_name = "sports33_test"

    else:

        raise ValueError(
            "sample debe ser 'train' o 'test'"
        )


    df = pd.read_csv(metadata_file)


    print("\n==============================")
    print(f"Muestra: {sample}")
    print("==============================")

    print(
        "Cantidad registros:",
        len(df)
    )


    # ----------------------------------------
    # Validar captions
    # ----------------------------------------

    if "captions" not in df.columns:

        raise ValueError(
            "No existe la columna 'captions'"
        )

    
    # ----------------------------------------
    # Separar captions
    # ----------------------------------------
    
    caption_records = []
    
    for _, row in df.iterrows():
    
        captions_list = (
            str(row["captions"])
            .split(simbolo_separacion)
        )
    
        for caption_id, caption in enumerate(captions_list):
    
            caption = caption.strip()
    
            if caption:
    
                caption_records.append(
                    {
                        "caption_index": len(caption_records),
                        "video_id": row["video_id"],
                        "sport": row["sport"],
                        "caption_id": caption_id,
                        "caption": caption
                    }
                )
    
    
    caption_df = pd.DataFrame(
        caption_records
    )
    
    if caption_df.empty:
        raise ValueError(
            "No se encontraron captions después de la separación"
        )
    captions = caption_df["caption"].tolist()


    embeddings = []


    # ----------------------------------------
    # Embeddings
    # ----------------------------------------

    model.eval()

    with torch.no_grad():

        for caption in tqdm(
            captions,
            desc=f"Embeddings {sample}"
        ):

            inputs = processor(
                text=[caption],
                return_tensors="pt",
                padding=True,
                truncation=True
            )

            inputs = {
                k: v.to(DEVICE)
                for k, v in inputs.items()
            }

            text_features = model.get_text_features(
                **inputs
            )

            text_features = (
                text_features /
                text_features.norm(
                    dim=-1,
                    keepdim=True
                )
            )

            embeddings.append(
                text_features.cpu().numpy()
            )


    embeddings = np.vstack(
        embeddings
    )


    # ----------------------------------------
    # Guardar
    # ----------------------------------------

    np.save(
        OUTPUT_EMBEDDINGS /
        f"{output_name}_caption_embeddings.npy",
        embeddings
    )


    caption_df.to_csv(
        OUTPUT_EMBEDDINGS /
        f"{output_name}_caption_metadata.csv",
        index=False
    )


    print(
        "Shape:",
        embeddings.shape
    )


    return embeddings

In [6]:
# ============================================
# Copiar metadata al directorio de trabajo
# ============================================

from shutil import copy2

SOURCE_OUTPUTS = ROOT.parent / "outputs"

TRAIN_SOURCE = SOURCE_OUTPUTS / "sports33_msr-vtt_train.csv"
TEST_SOURCE = SOURCE_OUTPUTS / "sports33_msr-vtt_test.csv"

TRAIN_DEST = OUTPUTS / "sports33_msr-vtt_train.csv"
TEST_DEST = OUTPUTS / "sports33_msr-vtt_test.csv"

copy2(TRAIN_SOURCE, TRAIN_DEST)
copy2(TEST_SOURCE, TEST_DEST)

print("Archivos copiados:")
print(TRAIN_DEST)
print(TEST_DEST)

Archivos copiados:
/workspace/Exposicion3/notebook_MSR_VTT/outputs/sports33_msr-vtt_train.csv
/workspace/Exposicion3/notebook_MSR_VTT/outputs/sports33_msr-vtt_test.csv


In [16]:
# ============================================
# Copiar frames (solo si no existen)
# ============================================

from shutil import copytree

SOURCE_DATA = ROOT.parent / "data" / "msr-vtt"
DEST_DATA = DATA / "msr-vtt"

DEST_DATA.mkdir(
    parents=True,
    exist_ok=True
)

for folder in [
    "frames",
    "frames-test"
]:

    source = SOURCE_DATA / folder
    dest = DEST_DATA / folder

    if not source.exists():
        print(f"No existe el origen: {source}")
        continue

    if dest.exists():
        print(f"Ya existe: {dest}")
        continue

    copytree(
        source,
        dest
    )

    print(f"Copiado: {folder}")

print("\nProceso finalizado.")

Ya existe: /workspace/Exposicion3/notebook_MSR_VTT/data/msr-vtt/frames
Copiado: frames-test

Proceso finalizado.


In [7]:
# ============================================
# Cargar metadata train y test
# ============================================

import pandas as pd


TRAIN_CSV = ROOT / CONFIG.train_metadata

TEST_CSV = ROOT / CONFIG.test_metadata


df_train = pd.read_csv(
    TRAIN_CSV
)


df_test = pd.read_csv(
    TEST_CSV
)


print("Train:")
print(df_train.shape)

print("\nTest:")
print(df_test.shape)


print("\nColumnas:")
print(df_train.columns.tolist())

Train:
(132, 9)

Test:
(40, 9)

Columnas:
['sport', 'score', 'video_id', 'video', 'url', 'start_time', 'end_time', 'category', 'captions']


In [13]:
from transformers import CLIPModel, CLIPProcessor

model = CLIPModel.from_pretrained(
    CONFIG.clip_model_name
).to(DEVICE)

processor = CLIPProcessor.from_pretrained(
    CONFIG.clip_model_name
)

model.eval()

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

/usr/local/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPSdpaAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e

In [27]:
SEPARADOR = "|||"

caption_embeddings_train = build_caption_embeddings(
    "train",
    model,
    processor,
    simbolo_separacion=SEPARADOR
)


caption_embeddings_test = build_caption_embeddings(
    "test",
    model,
    processor,
    simbolo_separacion=SEPARADOR
)


Muestra: train
Cantidad registros: 132


Embeddings train: 100%|█████████████████████| 2640/2640 [01:10<00:00, 37.60it/s]


Shape: (2640, 512)

Muestra: test
Cantidad registros: 40


Embeddings test: 100%|████████████████████████| 800/800 [00:25<00:00, 31.14it/s]


Shape: (800, 512)


Función para obtener embeddings de los frames 

In [17]:
# ============================================
# Función: construir embeddings de frames
# ============================================

from pathlib import Path
from PIL import Image
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm


def build_frame_embeddings(
    sample,
    context,
    model,
    processor
):
    """
    Genera embeddings CLIP para todos los frames.

    Parámetros
    ----------
    sample : str
        "train" o "test"

    context : str
        Ejemplo:
            "context_12"
            "context_8"
            "context_4"
    """

    # ============================================
    # Seleccionar carpeta de frames
    # ============================================

    if CONFIG.split == "train":

        frame_root = (
            DATA /
            "msr-vtt" /
            "frames" /
            CONFIG.frame_context
        )

        output_prefix = (
            f"sports33_train_{CONFIG.frame_context}"
        )
    
    elif CONFIG.split == "test":
    
        frame_root = (
            DATA /
            "msr-vtt" /
            "frames-test" /
            CONFIG.frame_context
        )
    
        output_prefix = (
            f"sports33_test_{CONFIG.frame_context}"
        )

    else:
    
        raise ValueError(
            f"Split no válido: {CONFIG.split}"
        )
    
            output_prefix = f"sports33_test_{CONFIG.frame_context}"

 


    if not frame_root.exists():

        raise FileNotFoundError(
            frame_root
        )


    print("\n==============================")
    print(f"Muestra : {sample}")
    print(f"Contexto: {context}")
    print("==============================\n")


    embeddings = []

    metadata = []


    sports = sorted(

        folder

        for folder in frame_root.iterdir()

        if folder.is_dir()

    )


    model.eval()


    with torch.no_grad():

        for sport_dir in sports:

            images = sorted(

                sport_dir.glob("*.jpg")

            )

            print(
                f"{sport_dir.name}: {len(images)} frames"
            )


            for image_path in tqdm(
                images,
                leave=False
            ):

                image = Image.open(
                    image_path
                ).convert("RGB")


                inputs = processor(

                    images=image,

                    return_tensors="pt"

                )


                inputs = {

                    k: v.to(DEVICE)

                    for k, v in inputs.items()

                }


                image_features = model.get_image_features(
                    **inputs
                )


                image_features = (

                    image_features /

                    image_features.norm(
                        dim=-1,
                        keepdim=True
                    )

                )


                embeddings.append(
                    image_features.cpu().numpy()
                )


                video_id, frame_number = (
                    image_path.stem.rsplit(
                        "_frame_",
                        1
                    )
                )


                metadata.append({

                    "video_id": video_id,

                    "sport": sport_dir.name,

                    "frame_file": image_path.name,

                    "frame_number": int(
                        frame_number
                    )

                })


    embeddings = np.vstack(
        embeddings
    )


    metadata = pd.DataFrame(
        metadata
    )


    embedding_file = (

        OUTPUT_EMBEDDINGS /

        f"{output_prefix}_frame_embeddings.npy"

    )


    metadata_file = (

        OUTPUT_EMBEDDINGS /

        f"{output_prefix}_frame_metadata.csv"

    )


    np.save(

        embedding_file,

        embeddings

    )


    metadata.to_csv(

        metadata_file,

        index=False

    )


    print("\nEmbeddings:", embeddings.shape)

    print(
        "Guardado:",
        embedding_file
    )

    print(
        "Metadata:",
        metadata_file
    )


    return embeddings, metadata

In [18]:
# ============================================
# LLamadas para Embeddings train
# ============================================

frame_embeddings_train, frame_metadata_train = build_frame_embeddings(
    sample="train",
    context="context_12",
    model=model,
    processor=processor
)


Muestra : train
Contexto: context_12

basketball: 396 frames


soccer: 396 frames


swimming: 384 frames


tennis: 396 frames



Embeddings: (1572, 512)
Guardado: /workspace/Exposicion3/notebook_MSR_VTT/outputs/embeddings/sports33_train_context_12_frame_embeddings.npy
Metadata: /workspace/Exposicion3/notebook_MSR_VTT/outputs/embeddings/sports33_train_context_12_frame_metadata.csv


In [19]:
# ============================================
# Embeddings test
# ============================================

frame_embeddings_test, frame_metadata_test = build_frame_embeddings(
    sample="test",
    context="context_12",
    model=model,
    processor=processor
)


Muestra : test
Contexto: context_12

basketball: 60 frames


soccer: 60 frames


swimming: 36 frames


tennis: 60 frames



Embeddings: (216, 512)
Guardado: /workspace/Exposicion3/notebook_MSR_VTT/outputs/embeddings/sports33_test_context_12_frame_embeddings.npy
Metadata: /workspace/Exposicion3/notebook_MSR_VTT/outputs/embeddings/sports33_test_context_12_frame_metadata.csv


In [29]:
# ============================================
# Función: ranking CLIP por video
# ============================================

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F


def build_clip_frame_ranking(
    sample,
    context,
    top_k=None
):
    """
    Ranking CLIP entre caption y frames
    del mismo video.

    Parameters
    ----------
    sample : str
        "train" o "test"

    context : str
        Ejemplo:
        context_12

    top_k : int
        Número de frames a recuperar.
    """

    if top_k is None:
        top_k = CONFIG.top_k_frames


    # ============================================
    # Cargar embeddings
    # ============================================

    if sample == "train":

        caption_embeddings = np.load(
            OUTPUT_EMBEDDINGS /
            "sports33_train_caption_embeddings.npy"
        )

        frame_embeddings = np.load(
            OUTPUT_EMBEDDINGS /
            f"sports33_train_{context}_frame_embeddings.npy"
        )

        caption_metadata = pd.read_csv(
            OUTPUT_EMBEDDINGS /
            "sports33_train_caption_metadata.csv"
        )

        frame_metadata = pd.read_csv(
            OUTPUT_EMBEDDINGS /
            f"sports33_train_{context}_frame_metadata.csv"
        )

        output_file = (
            OUTPUT_TABLES /
            f"sports33_train_{context}_clip_ranking.csv"
        )

    elif sample == "test":

        caption_embeddings = np.load(
            OUTPUT_EMBEDDINGS /
            "sports33_test_caption_embeddings.npy"
        )

        frame_embeddings = np.load(
            OUTPUT_EMBEDDINGS /
            f"sports33_test_{context}_frame_embeddings.npy"
        )

        caption_metadata = pd.read_csv(
            OUTPUT_EMBEDDINGS /
            "sports33_test_caption_metadata.csv"
        )

        frame_metadata = pd.read_csv(
            OUTPUT_EMBEDDINGS /
            f"sports33_test_{context}_frame_metadata.csv"
        )

        output_file = (
            OUTPUT_TABLES /
            f"sports33_test_{context}_clip_ranking.csv"
        )

    else:

        raise ValueError(
            "sample debe ser train o test"
        )


    # ============================================
    # Tensor
    # ============================================

    caption_embeddings = torch.tensor(
        caption_embeddings,
        dtype=torch.float32
    )

    frame_embeddings = torch.tensor(
        frame_embeddings,
        dtype=torch.float32
    )


    ranking_rows = []


    # ============================================
    # Ranking por video
    # ============================================

    for caption_idx, row in caption_metadata.iterrows():

        video_id = row["video_id"]

        caption = row["caption"]
     
        caption_id = row["caption_id"]
        # Frames del mismo video
        frames_video = frame_metadata[
            frame_metadata.video_id == video_id
        ]


        if len(frames_video) == 0:

            print(
                f"No hay frames para {video_id}"
            )

            continue


        frame_indices = frames_video.index.tolist()


        frame_vectors = frame_embeddings[
            frame_indices
        ]


        caption_vector = caption_embeddings[
            caption_idx
        ].unsqueeze(0)


        similarity = F.cosine_similarity(

            caption_vector,

            frame_vectors,

            dim=1

        )


        values, indices = torch.topk(

            similarity,

            k=min(
                top_k,
                len(frames_video)
            )

        )


        for rank, (
            score,
            local_idx
        ) in enumerate(

            zip(
                values.tolist(),
                indices.tolist()
            ),

            start=1

        ):

            frame = frames_video.iloc[
                local_idx
            ]


            ranking_rows.append({

                "video_id": video_id,

                "sport": frame["sport"],

                "caption_id": caption_id,

                "caption": caption,

                "rank": rank,

                "similarity": score,

                "frame_file": frame["frame_file"],

                "frame_number": frame["frame_number"]

            })


    ranking = pd.DataFrame(
        ranking_rows
    )

    # ============================================
    # Mejor caption encontrado por video
    # ============================================
    
    best_frame_caption_match = (
        ranking
        .sort_values(
            "similarity",
            ascending=False
        )
        .groupby(
            "video_id"
        )
        .first()
        .reset_index()
    )
    ranking.to_csv(

        output_file,

        index=False

    )
    best_frame_caption_match.to_csv(
        OUTPUT_TABLES /
        f"{sample}_{context}_best_frame_caption_match.csv",
        index=False
    )

    print(
        f"\nRanking guardado:\n{output_file}"
    )

    print(
        f"Filas: {len(ranking)}"
    )
    print(
        "\nMejor similitud promedio por frame-caption:",
        best_frame_caption_match["similarity"].mean()
    )

    return ranking

In [31]:
#llamadas RANKING
ranking_train = build_clip_frame_ranking(
    sample="train",
    context="context_12"
)
ranking_test = build_clip_frame_ranking(
    sample="test",
    context="context_12"
)

No hay frames para video4759
No hay frames para video4759
No hay frames para video4759
No hay frames para video4759
No hay frames para video4759
No hay frames para video4759
No hay frames para video4759
No hay frames para video4759
No hay frames para video4759
No hay frames para video4759
No hay frames para video4759
No hay frames para video4759
No hay frames para video4759
No hay frames para video4759
No hay frames para video4759
No hay frames para video4759
No hay frames para video4759
No hay frames para video4759
No hay frames para video4759
No hay frames para video4759

Ranking guardado:
/workspace/Exposicion3/notebook_MSR_VTT/outputs/tables/sports33_train_context_12_clip_ranking.csv
Filas: 20960

Mejor similitud promedio por frame-caption: 0.35036182767562285
No hay frames para video1084
No hay frames para video1084
No hay frames para video1084
No hay frames para video1084
No hay frames para video1084
No hay frames para video1084
No hay frames para video1084
No hay frames para vid

In [39]:
# ============================================
# Generador BLIP captions por frame
# ============================================

from typing import Sequence, List

import pandas as pd
import torch

from PIL import Image
from tqdm import tqdm


def generate_blip_captions(
    image_paths: Sequence[str],
    batch_size: int
) -> List[str]:
    """
    Genera captions usando BLIP para una lista de frames.
    """

    from transformers import (
        BlipProcessor,
        BlipForConditionalGeneration
    )


    processor = BlipProcessor.from_pretrained(
        CONFIG.blip_model_name
    )


    model = BlipForConditionalGeneration.from_pretrained(
        CONFIG.blip_model_name
    ).to(DEVICE)


    model.eval()


    captions = []


    for start in tqdm(
        range(
            0,
            len(image_paths),
            batch_size
        ),
        desc="Generando captions BLIP"
    ):


        batch_paths = image_paths[
            start:start + batch_size
        ]


        images = [
            Image.open(path)
            .convert("RGB")
            for path in batch_paths
        ]


        inputs = processor(
            images=images,
            return_tensors="pt",
            padding=True
        )


        inputs = {
            k: v.to(DEVICE)
            for k, v in inputs.items()
        }


        with torch.no_grad():

            generated = model.generate(
                **inputs,
                max_new_tokens=32
            )


        decoded = processor.batch_decode(
            generated,
            skip_special_tokens=True
        )


        captions.extend(
            [
                x.strip()
                for x in decoded
            ]
        )


    return captions

In [41]:
train_frame_metadata = pd.read_csv(
    OUTPUT_EMBEDDINGS /
    "sports33_train_context_12_frame_metadata.csv"
)

In [72]:
# ============================================
# Construir CSV captions BLIP por frame
# ============================================

def build_blip_caption_csv(
    frame_metadata,
    output_file,
    batch_size,
    exclude_sport="golf"
):
    """
    Genera captions BLIP por frame.

    Entrada:
        video_id
        sport
        frame_file
        frame_number

    Salida:
        video_id
        frame_file
        sport
        caption_by_blip
    """


    df = frame_metadata.copy()


    # ----------------------------------------
    # Eliminar golf
    # ----------------------------------------

    df = df[
        df["sport"]
        .str.lower()
        != exclude_sport.lower()
    ].copy()


    print(
        "Frames después de eliminar golf:",
        len(df)
    )


    print(
        df["sport"].value_counts()
    )

    # ----------------------------------------
    # Construir ruta de cada frame
    # ----------------------------------------
    if CONFIG.split == "train":
        frame_root = (
            ROOT /
            "data" /
            "msr-vtt" /
            "frames" /
            "context_12"
        )
    elif CONFIG.split =="test":
        frame_root = (
            ROOT /
            "data" /
            "msr-vtt" /
            "frames-test" /
            "context_12"
        )
    else:
        raise ValueError(
            f"Split no válido: {CONFIG.split}"
        )
        
    df["image_path"] = df.apply(
        lambda row: str(
            frame_root /
            row["sport"] /
            row["frame_file"]
        ),
        axis=1
    )

    # Verificación primera imagen

    print(
        "\nPrimera imagen:"
    )

    print(
        df["image_path"].iloc[0]
    )


    # ----------------------------------------
    # Generar captions BLIP
    # ----------------------------------------

    captions = generate_blip_captions(
        df["image_path"].tolist(),
        batch_size
    )


    df["caption_by_blip"] = captions


    # ----------------------------------------
    # Guardar CSV
    # ----------------------------------------

    blip_df = df[
        [
            "video_id",
            "frame_file",
            "sport",
            "frame_number",
            "caption_by_blip"
        ]
    ]


    blip_df.to_csv(
        output_file,
        index=False
    )


    print(
        "\nCSV BLIP guardado:"
    )

    print(
        output_file
    )


    return blip_df

In [47]:
train_frame_metadata = pd.read_csv(
    OUTPUT_EMBEDDINGS /
    "sports33_train_context_12_frame_metadata.csv"
)


train_blip_df = build_blip_caption_csv(
    frame_metadata=train_frame_metadata,

    output_file=
        OUTPUT_TABLES /
        "sports33_train_context_12_blip_captions.csv",

    batch_size=CONFIG.batch_size,

    exclude_sport="golf"
)

Frames después de eliminar golf: 1572
sport
basketball    396
soccer        396
tennis        396
swimming      384
Name: count, dtype: int64

Primera imagen:
/workspace/Exposicion3/notebook_MSR_VTT/data/msr-vtt/frames/context_12/basketball/video1623_frame_000.jpg


/usr/local/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Generando captions BLIP: 100%|████████████████| 99/99 [2:02:09<00:00, 74.04s/it]



CSV BLIP guardado:
/workspace/Exposicion3/notebook_MSR_VTT/outputs/tables/sports33_train_context_12_blip_captions.csv


In [73]:
CONFIG.split = "test"
test_frame_metadata = pd.read_csv(
    OUTPUT_EMBEDDINGS /
    "sports33_test_context_12_frame_metadata.csv"
)


test_blip_df = build_blip_caption_csv(
    frame_metadata=test_frame_metadata,

    output_file=
        OUTPUT_TABLES /
        "sports33_test_context_12_blip_captions.csv",

    batch_size=CONFIG.batch_size,

    exclude_sport="golf"
)

Frames después de eliminar golf: 216
sport
basketball    60
soccer        60
tennis        60
swimming      36
Name: count, dtype: int64

Primera imagen:
/workspace/Exposicion3/notebook_MSR_VTT/data/msr-vtt/frames-test/context_12/basketball/video4038_frame_000.jpg


/usr/local/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Generando captions BLIP: 100%|██████████████████| 14/14 [14:20<00:00, 61.46s/it]



CSV BLIP guardado:
/workspace/Exposicion3/notebook_MSR_VTT/outputs/tables/sports33_test_context_12_blip_captions.csv


In [75]:
blip_test = pd.read_csv(
    OUTPUT_TABLES /
    "sports33_test_context_12_blip_captions.csv"
)

print(blip_test.head())

print("\nFilas:", len(blip_test))

print("\nVideos:", blip_test["video_id"].nunique())

    video_id               frame_file       sport  frame_number  \
0  video4038  video4038_frame_000.jpg  basketball             0   
1  video4038  video4038_frame_001.jpg  basketball             1   
2  video4038  video4038_frame_002.jpg  basketball             2   
3  video4038  video4038_frame_003.jpg  basketball             3   
4  video4038  video4038_frame_004.jpg  basketball             4   

                                     caption_by_blip  
0            a basketball game with a crowd watching  
1  kobe kobe, lakers, and kobe, lakers, kobe kobe...  
2                       kobe griffin vs kobe griffin  
3        a basketball player is running on the court  
4       a basketball player is standing on the court  

Filas: 216

Videos: 18


In [48]:
# =====================================================
# CNN Visual Encoder (ResNet50)
# =====================================================

import torch
import torch.nn as nn

from torchvision import models
from torchvision import transforms

from PIL import Image


# ---------------------------------------------
# Modelo
# ---------------------------------------------

resnet = models.resnet50(
    weights=models.ResNet50_Weights.DEFAULT
)

# eliminar la capa de clasificación
cnn_encoder = nn.Sequential(
    *list(resnet.children())[:-1]
)

cnn_encoder = cnn_encoder.to(DEVICE)
cnn_encoder.eval()


# ---------------------------------------------
# Transformaciones
# ---------------------------------------------

transform = transforms.Compose([

    transforms.Resize(256),

    transforms.CenterCrop(224),

    transforms.ToTensor(),

    transforms.Normalize(

        mean=[0.485,0.456,0.406],

        std=[0.229,0.224,0.225]

    )

])

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████████████████████████████████| 97.8M/97.8M [00:05<00:00, 19.4MB/s]


Construcción de 12 tokens visuales
03_visual_tokens.py

Este script será ejecutado una sola vez.

Su objetivo es generar los embeddings CNN para cada uno de los 12 frames.

Video760

↓

Frame000

↓

CNN

↓

2048

Frame001

↓

CNN

↓

2048

...

↓

Frame011

↓

CNN

↓

2048

↓

Tensor (12,2048)

↓

Guardar

In [50]:
import numpy as np
import pandas as pd

from PIL import Image

import torch
from tqdm import tqdm
# =====================================================
# Construir tokens visuales
# =====================================================
def build_visual_tokens(
    sample,
    context,
    cnn_encoder,
    transform
):

    if sample == "train":

        metadata = pd.read_csv(

            OUTPUT_EMBEDDINGS /
            f"sports33_train_{context}_frame_metadata.csv"

        )

        output_name = (
            OUTPUT_EMBEDDINGS /
            "sports33_train_visual_tokens.npy"
        )

    elif sample == "test":

        metadata = pd.read_csv(

            OUTPUT_EMBEDDINGS /
            f"sports33_test_{context}_frame_metadata.csv"

        )

        output_name = (
            OUTPUT_EMBEDDINGS /
            "sports33_test_visual_tokens.npy"
        )

    else:

        raise ValueError()

pipeline: extracción de representaciones visuales por frame usando ResNet50.
Frames
(context_12)
      |
      v
ResNet50
      |
      v
visual_tokens_context_12.npy
      |
      v
+ Text embeddings / captions
      |
      v
Dataset multimodal
      |
      v
VisualBERT / LXMERT
      |
      v
Ranking texto ↔ frames/videos
      |
      v
Evaluación + perturbaciones

El siguiente paso lógico es construir el dataset multimodal.
Paso 1  generar embeddings de texto, enfocandonos en convertir tus captions a vectores.
Para VisualBERT/LXMERT necesitamos:

tokens de texto (input_ids)
máscara de atención (attention_mask)

usando el tokenizer de BERT.

In [74]:
# =====================================================
# Construcción de visual tokens por video
# =====================================================

import pandas as pd
import numpy as np
from tqdm import tqdm
from PIL import Image

CONFIG.split = "test"
# Cargar metadata de frames
if CONFIG.split == "train":

    CONFIG.frame_root = "data/msr-vtt/frames"

    CONFIG.frame_metadata = (
        "outputs/embeddings/"
        "sports33_train_context_12_frame_metadata.csv"
    )

elif CONFIG.split == "test":

    CONFIG.frame_root = "data/msr-vtt/frames-test"

    CONFIG.frame_metadata = (
        "outputs/embeddings/"
        "sports33_test_context_12_frame_metadata.csv"
    )

else:

    raise ValueError(
        f"Split no válido: {CONFIG.split}"
    )


metadata = pd.read_csv(
    ROOT / CONFIG.frame_metadata
)


visual_tokens = {}


video_ids = sorted(
    metadata["video_id"].unique()
)


for video_id in tqdm(
    video_ids,
    desc="Construyendo visual tokens"
):

    # ---------------------------------------------
    # Frames del video
    # ---------------------------------------------

    frames = (
        metadata[
            metadata["video_id"] == video_id
        ]
        .sort_values("frame_number")
    )


    if len(frames) != CONFIG.frames_per_video:

        print(
            f"Advertencia: {video_id} tiene "
            f"{len(frames)} frames."
        )


    embeddings = []


    # ---------------------------------------------
    # Embedding CNN de cada frame
    # ---------------------------------------------

    for _, row in frames.iterrows():

        image_path = (
            ROOT /
            CONFIG.frame_root /
            CONFIG.frame_context /
            row["sport"] /
            row["frame_file"]
        )


        image = (
            Image.open(image_path)
            .convert("RGB")
        )


        image = transform(image)

        image = (
            image
            .unsqueeze(0)
            .to(DEVICE)
        )


        with torch.no_grad():

            feature = cnn_encoder(image)


        feature = feature.squeeze()


        embeddings.append(
            feature.cpu().numpy()
        )


    # ---------------------------------------------
    # Guardar tokens del video
    # ---------------------------------------------

    visual_tokens[video_id] = np.stack(
        embeddings
    )



# =====================================================
# Guardar visual tokens
# =====================================================

if CONFIG.split == "train":

    output_name = (
        OUTPUT_EMBEDDINGS /
        f"visual_tokens_{CONFIG.frame_context}.npy"
    )

elif CONFIG.split == "test":

    output_name = (
        OUTPUT_EMBEDDINGS /
        f"visual_tokens_test_{CONFIG.frame_context}.npy"
    )

else:

    raise ValueError(
        f"Split no válido: {CONFIG.split}"
    )


np.save(
    output_name,
    visual_tokens,
    allow_pickle=True
)


print("\nVisual tokens guardados en:")
print(output_name)


print(
    "\nCantidad de videos:",
    len(visual_tokens)
)


primer_video = next(iter(visual_tokens))


print(
    "\nEjemplo:",
    primer_video
)


print(
    "Shape:",
    visual_tokens[primer_video].shape
)

Construyendo visual tokens: 100%|███████████████| 18/18 [00:36<00:00,  2.00s/it]


Visual tokens guardados en:
/workspace/Exposicion3/notebook_MSR_VTT/outputs/embeddings/visual_tokens_test_context_12.npy

Cantidad de videos: 18

Ejemplo: video21
Shape: (12, 2048)


In [56]:
# =====================================================
# Verificar shape de visual tokens
# =====================================================

print("Cantidad de videos:")
print(len(visual_tokens))


# tomar un video cualquiera
video_id = next(iter(visual_tokens))

print("\nVideo ejemplo:")
print(video_id)


print("\nShape de visual tokens:")
print(visual_tokens[video_id].shape)
metadata["video_id"].nunique()


train = pd.read_csv(
    ROOT / CONFIG.train_metadata
)

print(train.columns)
print(train.head(2))

Cantidad de videos:
131

Video ejemplo:
video1103

Shape de visual tokens:
(12, 2048)
Index(['sport', 'score', 'video_id', 'video', 'url', 'start_time', 'end_time',
       'category', 'captions'],
      dtype='object')
        sport  score   video_id          video  \
0  basketball     16  video1623  video1623.mp4   
1  basketball     16  video1667  video1667.mp4   

                                           url  start_time  end_time  \
0  https://www.youtube.com/watch?v=1IIDthMD09M      107.89    119.00   
1  https://www.youtube.com/watch?v=cpCrFn7MQ8k      200.45    213.49   

   category                                           captions  
0         3  a basketball game being played ||| a basketbal...  
1         1  a basketball player does a slam dunk ||| a man...  


In [58]:
# =====================================================
# 04_build_multimodal_dataset.py
# Parte 1
# Cargar datos
# =====================================================

import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from tqdm import tqdm

from transformers import AutoTokenizer


# =====================================================
# CONFIGURACIÓN
# =====================================================
CONFIG.split = "train"
print("=" * 60)
print("Construcción del Dataset Multimodal")
print("=" * 60)


# -----------------------------------------------------
# Tokenizer
# -----------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)


# -----------------------------------------------------
# Rutas
# -----------------------------------------------------
# =====================================================
# Seleccionar archivos según el split
# =====================================================

if CONFIG.split == "train":

    visual_tokens_file = (
        OUTPUT_EMBEDDINGS /
        f"visual_tokens_{CONFIG.frame_context}.npy"
    )

    ranking_file = (
        OUTPUT_TABLES /
        f"sports33_train_{CONFIG.frame_context}_clip_ranking.csv"
    )

    blip_file = (
        OUTPUT_TABLES /
        f"sports33_train_{CONFIG.frame_context}_blip_captions.csv"
    )

    output_dataset = (
        DATA_PROCESSED /
        f"multimodal_train_{CONFIG.frame_context}.pt"
    )


elif CONFIG.split == "test":

    visual_tokens_file = (
        OUTPUT_EMBEDDINGS /
        f"visual_tokens_test_{CONFIG.frame_context}.npy"
    )

    ranking_file = (
        OUTPUT_TABLES /
        f"sports33_test_{CONFIG.frame_context}_clip_ranking.csv"
    )

    blip_file = (
        OUTPUT_TABLES /
        f"sports33_test_{CONFIG.frame_context}_blip_captions.csv"
    )

    output_dataset = (
        DATA_PROCESSED /
        f"multimodal_test_{CONFIG.frame_context}.pt"
    )


else:

    raise ValueError(
        f"Split no válido: {CONFIG.split}"
    )

print("\nArchivos")

print(visual_tokens_file)
print(ranking_file)
print(blip_file)


# =====================================================
# Cargar visual tokens
# =====================================================

print("\nCargando visual tokens...")

visual_tokens = np.load(
    visual_tokens_file,
    allow_pickle=True
).item()

print(
    "Videos:",
    len(visual_tokens)
)


# =====================================================
# Cargar ranking CLIP
# =====================================================

print("\nCargando ranking...")

ranking = pd.read_csv(
    ranking_file
)

print(
    "Filas ranking:",
    len(ranking)
)

print(
    ranking.head()
)


# =====================================================
# Cargar captions BLIP
# =====================================================

print("\nCargando captions BLIP...")

blip = pd.read_csv(
    blip_file
)

print(
    "Filas BLIP:",
    len(blip)
)

print(
    blip.head()
)


# =====================================================
# Verificar columnas
# =====================================================

print("\nColumnas Ranking")

print(
    ranking.columns.tolist()
)

print("\nColumnas BLIP")

print(
    blip.columns.tolist()
)

Construcción del Dataset Multimodal

Archivos
/workspace/Exposicion3/notebook_MSR_VTT/outputs/embeddings/visual_tokens_context_12.npy
/workspace/Exposicion3/notebook_MSR_VTT/outputs/tables/sports33_train_context_12_clip_ranking.csv
/workspace/Exposicion3/notebook_MSR_VTT/outputs/tables/sports33_train_context_12_blip_captions.csv

Cargando visual tokens...
Videos: 131

Cargando ranking...
Filas ranking: 20960
    video_id       sport  caption_id                         caption  rank  \
0  video1623  basketball           0  a basketball game being played     1   
1  video1623  basketball           0  a basketball game being played     2   
2  video1623  basketball           0  a basketball game being played     3   
3  video1623  basketball           0  a basketball game being played     4   
4  video1623  basketball           0  a basketball game being played     5   

   similarity               frame_file  frame_number  
0    0.271185  video1623_frame_003.jpg             3  
1    0.26

In [76]:
# =====================================================
# Merge Ranking + BLIP + Visual Tokens
# =====================================================

import pandas as pd


def build_multimodal_metadata(
    ranking,
    blip,
    visual_tokens
):

    print("\n" + "=" * 60)
    print("Uniendo Ranking + BLIP + Visual Tokens")
    print("=" * 60)


    # -------------------------------------------------
    # Merge Ranking + BLIP
    # -------------------------------------------------

    multimodal = ranking.merge(

        blip,

        on=[
            "video_id",
            "frame_file",
            "sport",
            "frame_number"
        ],

        how="left"

    )


    print("\nFilas después del merge:")
    print(len(multimodal))


    print("\nColumnas:")
    print(
        multimodal.columns.tolist()
    )


    # -------------------------------------------------
    # Verificar captions BLIP faltantes
    # -------------------------------------------------

    missing = (
        multimodal["caption_by_blip"]
        .isna()
        .sum()
    )


    print("\nCaptions BLIP faltantes:")
    print(missing)



    # -------------------------------------------------
    # Verificar visual tokens disponibles
    # -------------------------------------------------

    video_ids_tokens = set(
        visual_tokens.keys()
    )


    video_ids_dataset = set(
        multimodal["video_id"]
    )


    faltantes = sorted(

        video_ids_dataset -
        video_ids_tokens

    )


    print("\nVideos sin visual tokens:")
    print(len(faltantes))


    if len(faltantes) > 0:

        print(faltantes[:20])



    # -------------------------------------------------
    # Conservar únicamente videos válidos
    # -------------------------------------------------

    multimodal = multimodal[
        multimodal["video_id"].isin(
            video_ids_tokens
        )
    ].copy()


    print("\nFilas finales:")
    print(len(multimodal))


    print(
        "\nVideos únicos:",
        multimodal["video_id"].nunique()
    )



    # -------------------------------------------------
    # Orden temporal
    # -------------------------------------------------

    multimodal = multimodal.sort_values(

        [
            "video_id",
            "frame_number"
        ]

    ).reset_index(drop=True)



    # -------------------------------------------------
    # Referencia para recuperar tokens
    # -------------------------------------------------

    multimodal["visual_token_key"] = (
        multimodal["video_id"]
    )


    print("\nPrimeras filas:")

    display(
        multimodal.head()
    )


    return multimodal

In [79]:
# =====================================================
# Cargar Ranking + BLIP + Visual Tokens -cargar archivos a utilizar
# =====================================================

import pandas as pd
import numpy as np


# -----------------------------------------------------
# Cargar Visual Tokens
# -----------------------------------------------------

train_visual_tokens = np.load(
    OUTPUT_EMBEDDINGS /
    f"visual_tokens_{CONFIG.frame_context}.npy",
    allow_pickle=True
).item()


test_visual_tokens = np.load(
    OUTPUT_EMBEDDINGS /
    f"visual_tokens_test_{CONFIG.frame_context}.npy",
    allow_pickle=True
).item()


print("Visual tokens cargados")
print(
    "Train:",
    len(train_visual_tokens)
)

print(
    "Test:",
    len(test_visual_tokens)
)



# -----------------------------------------------------
# Cargar BLIP captions
# -----------------------------------------------------

train_blip_file = (
    OUTPUT_TABLES /
    f"sports33_train_{CONFIG.frame_context}_blip_captions.csv"
)


test_blip_file = (
    OUTPUT_TABLES /
    f"sports33_test_{CONFIG.frame_context}_blip_captions.csv"
)



train_blip_df = pd.read_csv(
    train_blip_file
)


test_blip_df = pd.read_csv(
    test_blip_file
)



print("\nBLIP cargado")

print(
    "Train BLIP:",
    len(train_blip_df)
)

print(
    "Test BLIP:",
    len(test_blip_df)
)



# -----------------------------------------------------
# Cargar Ranking
# -----------------------------------------------------

train_ranking_file = (
    OUTPUT_TABLES /
    f"sports33_train_{CONFIG.frame_context}_clip_ranking.csv"
)


test_ranking_file = (
    OUTPUT_TABLES /
    f"sports33_test_{CONFIG.frame_context}_clip_ranking.csv"
)



ranking_train = pd.read_csv(
    train_ranking_file
)


ranking_test = pd.read_csv(
    test_ranking_file
)



print("\nRanking cargado")

print(
    "Train ranking:",
    len(ranking_train)
)

print(
    "Test ranking:",
    len(ranking_test)
)

Visual tokens cargados
Train: 131
Test: 18

BLIP cargado
Train BLIP: 1572
Test BLIP: 216

Ranking cargado
Train ranking: 20960
Test ranking: 2880


In [81]:
CONFIG.split = "train"
train_multimodal = build_multimodal_metadata(
    ranking_train,
    train_blip_df,
    train_visual_tokens
)


Uniendo Ranking + BLIP + Visual Tokens

Filas después del merge:
20960

Columnas:
['video_id', 'sport', 'caption_id', 'caption', 'rank', 'similarity', 'frame_file', 'frame_number', 'caption_by_blip']

Captions BLIP faltantes:
0

Videos sin visual tokens:
0

Filas finales:
20960

Videos únicos: 131

Primeras filas:


,video_id,sport,caption_id,caption,rank,similarity,frame_file,frame_number,caption_by_blip,visual_token_key
0,video1103,soccer,0,a man cooking food,1,0.185026,video1103_frame_000.jpg,0,a soccer game is being played on the field,video1103
1,video1103,soccer,1,a man is playing soccer,1,0.277449,video1103_frame_000.jpg,0,a soccer game is being played on the field,video1103
2,video1103,soccer,2,a man missing a goal in soccer with his team m...,1,0.300338,video1103_frame_000.jpg,0,a soccer game is being played on the field,video1103
3,video1103,soccer,3,a professional soccer game with music playing ...,2,0.281623,video1103_frame_000.jpg,0,a soccer game is being played on the field,video1103
4,video1103,soccer,4,a professional soccer game,2,0.285093,video1103_frame_000.jpg,0,a soccer game is being played on the field,video1103


In [82]:
CONFIG.split = "test"
test_multimodal = build_multimodal_metadata(
    ranking_test,
    test_blip_df,
    test_visual_tokens
)


Uniendo Ranking + BLIP + Visual Tokens

Filas después del merge:
2880

Columnas:
['video_id', 'sport', 'caption_id', 'caption', 'rank', 'similarity', 'frame_file', 'frame_number', 'caption_by_blip']

Captions BLIP faltantes:
0

Videos sin visual tokens:
0

Filas finales:
2880

Videos únicos: 18

Primeras filas:


,video_id,sport,caption_id,caption,rank,similarity,frame_file,frame_number,caption_by_blip,visual_token_key
0,video21,tennis,0,there is a man is talking about a product,1,0.232741,video21_frame_000.jpg,0,a tennis court with a green court and white lines,video21
1,video21,tennis,2,a man is describing how to properly grip a bad...,8,0.355760,video21_frame_001.jpg,1,a person is pointing at a tennis court,video21
2,video21,tennis,4,an asian man using a tennis racket at a court,5,0.284596,video21_frame_001.jpg,1,a person is pointing at a tennis court,video21
3,video21,tennis,6,man describes how to hold a raquet while a man...,7,0.303955,video21_frame_001.jpg,1,a person is pointing at a tennis court,video21
4,video21,tennis,7,a guy holds a racket and descibes how to use a...,4,0.376150,video21_frame_001.jpg,1,a person is pointing at a tennis court,video21



outputs/

a) embeddings/
 visual_tokens_context_12.npy
 visual_tokens_test_context_12.npy

b) tables/
 sports33_train_context_12_blip_captions.csv
 sports33_test_context_12_blip_captions.csv
 sports33_train_context_12_clip_ranking.csv
 sports33_test_context_12_clip_ranking.csv

c) datasets/
     sports33_train_context_12_multimodal.pt
     sports33_test_context_12_multimodal.pt

In [86]:
# =====================================================
# Parte 3
# Construcción Dataset Multimodal por VIDEO
# Secuencia de frames + BLIP captions
# =====================================================

import torch
import pandas as pd
from tqdm import tqdm


def build_video_multimodal_dataset(
    multimodal,
    visual_tokens,
    tokenizer,
    TEXT_SOURCE="caption_by_blip",
    MAX_LENGTH=40
):

    print("\n" + "=" * 60)
    print("Construyendo Dataset Multimodal por Video")
    print("=" * 60)


    dataset = []


    # -------------------------------------------------
    # Agrupar por video
    # -------------------------------------------------

    video_groups = multimodal.groupby(
        "video_id"
    )


    # -------------------------------------------------
    # Una muestra por video
    # -------------------------------------------------

    for video_id, group in tqdm(
        video_groups,
        desc="Creando dataset"
    ):


        # Orden temporal de frames

        group = group.sort_values(
            "frame_number"
        )


        # ---------------------------------------------
        # Visual tokens
        # ---------------------------------------------

        if video_id not in visual_tokens:

            continue


        visual = visual_tokens[video_id]


        visual = torch.tensor(
            visual,
            dtype=torch.float32
        )


        # ---------------------------------------------
        # Captions BLIP por frame
        # ---------------------------------------------

        captions = (
            group[TEXT_SOURCE]
            .dropna()
            .astype(str)
            .tolist()
        )


        if len(captions) == 0:

            continue


        # Unimos captions de la secuencia

        text = " ".join(
            captions
        )


        encoded = tokenizer(

            text,

            padding="max_length",

            truncation=True,

            max_length=MAX_LENGTH,

            return_tensors="pt"

        )


        # ---------------------------------------------
        # Información del video
        # ---------------------------------------------

        sample = {

            "video_id":
                video_id,


            "sport":
                group["sport"].iloc[0],


            "captions_blip":
                captions,


            "num_frames":
                len(group),


            "visual_embeds":
                visual,


            "input_ids":
                encoded["input_ids"].squeeze(0),


            "attention_mask":
                encoded["attention_mask"].squeeze(0)

        }


        dataset.append(sample)



    print("\nDataset construido")

    print(
        "Cantidad de videos:",
        len(dataset)
    )


    if len(dataset) > 0:

        print("\nEjemplo:")

        print(
            dataset[0]["video_id"]
        )

        print(
            "Visual:",
            dataset[0]["visual_embeds"].shape
        )

        print(
            "Frames texto:",
            len(dataset[0]["captions_blip"])
        )


    return dataset

In [87]:
CONFIG.split = "train"
train_dataset = build_video_multimodal_dataset(
    train_multimodal,
    train_visual_tokens,
    tokenizer
)


Construyendo Dataset Multimodal por Video


Creando dataset: 100%|████████████████████████| 131/131 [00:02<00:00, 45.40it/s]



Dataset construido
Cantidad de videos: 131

Ejemplo:
video1103
Visual: torch.Size([12, 2048])
Frames texto: 160


In [88]:
CONFIG.split = "test"
test_dataset = build_video_multimodal_dataset(
    test_multimodal,
    test_visual_tokens,
    tokenizer
)


Construyendo Dataset Multimodal por Video


Creando dataset: 100%|██████████████████████████| 18/18 [00:00<00:00, 50.69it/s]


Dataset construido
Cantidad de videos: 18

Ejemplo:
video21
Visual: torch.Size([12, 2048])
Frames texto: 160


In [89]:
print(dataset[0].keys())

dict_keys(['video_id', 'sport', 'caption', 'caption_by_blip', 'frame_file', 'frame_number', 'rank', 'similarity', 'visual_embeds', 'input_ids', 'attention_mask'])


In [90]:
sample = dataset[0]

print("Video:", sample["video_id"])
print("Deporte:", sample["sport"])

print("\nVisual Embeds:")
print(sample["visual_embeds"].shape)

print("\nInput IDs:")
print(sample["input_ids"].shape)

print("\nAttention Mask:")
print(sample["attention_mask"].shape)

print("\nCaption original:")
print(sample["caption"])

print("\nCaption BLIP:")
print(sample["caption_by_blip"])

Video: video1103
Deporte: soccer

Visual Embeds:
torch.Size([12, 2048])

Input IDs:
torch.Size([40])

Attention Mask:
torch.Size([40])

Caption original:
a man cooking food

Caption BLIP:
a soccer game is being played on the field


In [91]:
# =====================================================
# Parte 4
# Guardar Dataset Multimodal por Video
# =====================================================

import random
import torch


def save_multimodal_dataset(
    dataset,
    output_dataset
):

    print("\n" + "=" * 60)
    print("Guardando Dataset")
    print("=" * 60)


    # -------------------------------------------------
    # Guardar
    # -------------------------------------------------

    torch.save(
        dataset,
        output_dataset
    )


    print("\nDataset guardado en:")
    print(output_dataset)



    # =================================================
    # Estadísticas
    # =================================================

    print("\n" + "=" * 60)
    print("Resumen")
    print("=" * 60)


    print(
        "Número de muestras:",
        len(dataset)
    )


    print(
        "Número de videos:",
        len(
            set(
                d["video_id"]
                for d in dataset
            )
        )
    )



    # -------------------------------------------------
    # Distribución por deporte
    # -------------------------------------------------

    sports = {}


    for sample in dataset:

        sport = sample["sport"]

        sports[sport] = sports.get(
            sport,
            0
        ) + 1



    print("\nDistribución")


    for sport in sorted(sports):

        print(
            f"{sport:12s}: {sports[sport]}"
        )



    # =================================================
    # Validación
    # =================================================

    print("\n" + "=" * 60)
    print("Validación")
    print("=" * 60)


    sample = random.choice(dataset)



    print("\nVideo:")
    print(
        sample["video_id"]
    )


    print("\nSport:")
    print(
        sample["sport"]
    )


    print("\nNúmero de frames:")
    print(
        sample["num_frames"]
    )


    print("\nPrimeros captions BLIP:")

    for i, caption in enumerate(
        sample["captions_blip"][:3]
    ):

        print(
            i,
            ":",
            caption
        )


    print("\nVisual Embeds:")
    print(
        sample["visual_embeds"].shape
    )


    print("\nInput IDs:")
    print(
        sample["input_ids"].shape
    )


    print("\nAttention Mask:")
    print(
        sample["attention_mask"].shape
    )


    print(
        "\nDataset listo para VisualBERT / LXMERT"
    )

In [22]:
#Carpetas para outputs
from pathlib import Path

OUTPUT_DATASETS = ROOT / "outputs" / "datasets"

OUTPUT_DATASETS.mkdir(
    parents=True,
    exist_ok=True
)

print(OUTPUT_DATASETS)

/workspace/Exposicion3/notebook_MSR_VTT/outputs/datasets


In [94]:
CONFIG.split = "train"
save_multimodal_dataset(
    train_dataset,
    OUTPUT_DATASETS /
    "sports33_train_context_12_multimodal.pt"
)


Guardando Dataset

Dataset guardado en:
/workspace/Exposicion3/notebook_MSR_VTT/outputs/datasets/sports33_train_context_12_multimodal.pt

Resumen
Número de muestras: 131
Número de videos: 131

Distribución
basketball  : 33
soccer      : 33
swimming    : 32
tennis      : 33

Validación

Video:
video5197

Sport:
swimming

Número de frames:
160

Primeros captions BLIP:
0 : a small house in the middle of a lake
1 : a small house in the middle of a lake
2 : a small house in the middle of a lake

Visual Embeds:
torch.Size([12, 2048])

Input IDs:
torch.Size([40])

Attention Mask:
torch.Size([40])

Dataset listo para VisualBERT / LXMERT


In [95]:
CONFIG.split = "test"
save_multimodal_dataset(
    test_dataset,
    OUTPUT_DATASETS /
    "sports33_test_context_12_multimodal.pt"
)


Guardando Dataset

Dataset guardado en:
/workspace/Exposicion3/notebook_MSR_VTT/outputs/datasets/sports33_test_context_12_multimodal.pt

Resumen
Número de muestras: 18
Número de videos: 18

Distribución
basketball  : 5
soccer      : 5
swimming    : 3
tennis      : 5

Validación

Video:
video4918

Sport:
tennis

Número de frames:
160

Primeros captions BLIP:
0 : a man in a green shirt playing badminton
1 : a man in a green shirt playing badminton
2 : a man in a green shirt playing badminton

Visual Embeds:
torch.Size([12, 2048])

Input IDs:
torch.Size([40])

Attention Mask:
torch.Size([40])

Dataset listo para VisualBERT / LXMERT


In [ ]:
### **Congelar solo entrenar la capa final**
Entrenar solo la capa final (con VisualBERT y LXMERT congelados) tiene varias ventajas para tu caso:

 Es totalmente válido metodológicamente.
 Evita sobreajuste, ya que tu dataset es relativamente pequeño.
 Se puede ejecutar en CPU.
 La comparación entre ambos modelos es justa.
#### **Revisión cualitativa inicial**




In [66]:
from transformers import VisualBertModel

model = VisualBertModel.from_pretrained(
    CONFIG.visualbert_model
)

print(model.config)

config.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/448M [00:00<?, ?B/s]

VisualBertConfig {
  "_name_or_path": "uclanlp/visualbert-vqa-coco-pre",
  "architectures": [
    "VisualBertForPreTraining"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "bypass_transformer": false,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "visual_bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 1,
  "special_visual_initialize": true,
  "transformers_version": "4.44.2",
  "type_vocab_size": 2,
  "visual_embedding_dim": 2048,
  "vocab_size": 30522
}



In [67]:
print(model.config.visual_embedding_dim)

2048


La representación visual es compatible directamente con VisualBERT.

Tus visual_embeds son:

(batch_size, 12, 2048)

y VisualBERT espera exactamente:

(batch_size, num_visual_tokens, 2048)
PIPELINE
Caption
      │
      ▼
BERT Tokenizer
      │
      ▼
VisualBERT (congelado)
      ▲
      │
Visual Tokens
(12×2048)

      │
      ▼
Pooler Output (768)

      │
      ▼
Linear(768 → 4 deportes)

      │
      ▼
CrossEntropyLoss

Ahora prepararemos los datos para VisualBert
============================================
batch de 16 videos.
cada texto tiene longitud máxima 40 tokens.
visual_embeds:
torch.Size([16, 12, 2048])

Esta es la parte más importante:

16 videos en el batch.
12 frames por video.
cada frame representado por un embedding de 2048 dimensiones.

In [58]:
# =====================================================
# 05_train_visualbert.py
# Parte 1 - variables para reproducibilidad
# =====================================================

import random
import numpy as np
import torch
import torch.nn as nn

from tqdm import tqdm

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from transformers import VisualBertModel


# =====================================================
# Reproducibilidad
# =====================================================

torch.manual_seed(CONFIG.seed)
np.random.seed(CONFIG.seed)
random.seed(CONFIG.seed)

DEVICE = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else

    "cpu"

)

print(DEVICE)


# =====================================================
# Cargar dataset
# =====================================================

dataset_path = (
    OUTPUT_DATASETS /
    f"sports33_train_{CONFIG.frame_context}_multimodal.pt"
)

dataset = torch.load(

    dataset_path,

    weights_only=False

)

print(

    "Número de muestras:",

    len(dataset)

)


# =====================================================
# Deportes
# =====================================================

sports = sorted(

    list(

        set(

            d["sport"]

            for d in dataset

        )

    )

)

print(sports)


label2id = {

    sport: idx

    for idx, sport

    in enumerate(sports)

}

id2label = {

    idx: sport

    for sport, idx

    in label2id.items()

}

print(label2id)


NUM_CLASSES = len(

    label2id

)

print(

    "Número de clases:",

    NUM_CLASSES

)


# =====================================================
# Dataset PyTorch
# =====================================================

class SportsDataset(

    Dataset

):

    def __init__(

        self,

        samples

    ):

        self.samples = samples


    def __len__(

        self

    ):

        return len(

            self.samples

        )


    def __getitem__(

        self,

        idx

    ):

        sample = self.samples[idx]

        return {

            "input_ids":

                sample["input_ids"],

            "attention_mask":

                sample["attention_mask"],

            "visual_embeds":

                sample["visual_embeds"],

            "label":

                torch.tensor(

                    label2id[

                        sample["sport"]

                    ],

                    dtype=torch.long

                ),

            "video_id":

                sample["video_id"]

        }


# =====================================================
# Crear Dataset
# =====================================================

full_dataset = SportsDataset(

    dataset

)

print(

    len(full_dataset)

)

cpu
Número de muestras: 131
['basketball', 'soccer', 'swimming', 'tennis']
{'basketball': 0, 'soccer': 1, 'swimming': 2, 'tennis': 3}
Número de clases: 4
131


In [59]:
# =====================================================
# Train / Validation
# =====================================================

from torch.utils.data import random_split


train_size = int(

    0.8 *

    len(full_dataset)

)

val_size = (

    len(full_dataset)

    -

    train_size

)


train_dataset, val_dataset = random_split(

    full_dataset,

    [

        train_size,

        val_size

    ],

    generator=torch.Generator().manual_seed(

        CONFIG.seed

    )

)


print(

    len(train_dataset),

    len(val_dataset)

)


# =====================================================
# DataLoader
# =====================================================

train_loader = DataLoader(

    train_dataset,

    batch_size=CONFIG.batch_size,

    shuffle=True

)

val_loader = DataLoader(

    val_dataset,

    batch_size=CONFIG.batch_size,

    shuffle=False

)


# =====================================================
# Verificación
# =====================================================

batch = next(

    iter(

        train_loader

    )

)

print(

    batch["input_ids"].shape

)

print(

    batch["visual_embeds"].shape

)

print(

    batch["label"].shape

)

104 27
torch.Size([16, 40])
torch.Size([16, 12, 2048])
torch.Size([16])


In [60]:
# =====================================================
# Parte 2
# Modelo VisualBERT para clasificación deportiva
# =====================================================

import torch.nn as nn
from transformers import VisualBertModel


# =====================================================
# Cargar VisualBERT
# =====================================================

visualbert = VisualBertModel.from_pretrained(
    CONFIG.visualbert_model
)


# =====================================================
# Clasificador
# =====================================================

class VisualBertSportsClassifier(
    nn.Module
):

    def __init__(
        self,
        visualbert,
        num_classes
    ):

        super().__init__()

        self.visualbert = visualbert


        hidden_size = (
            visualbert.config.hidden_size
        )


        self.classifier = nn.Linear(
            hidden_size,
            num_classes
        )


    def forward(
        self,
        input_ids,
        attention_mask,
        visual_embeds
    ):


        outputs = self.visualbert(

            input_ids=input_ids,

            attention_mask=attention_mask,

            visual_embeds=visual_embeds

        )


        # representación global

        pooled_output = (
            outputs.pooler_output
        )


        logits = self.classifier(
            pooled_output
        )


        return logits

In [61]:
# =====================================================
# Instanciar modelo
# =====================================================

model = VisualBertSportsClassifier(
    visualbert,
    NUM_CLASSES
)


model = model.to(
    DEVICE
)


print(model)

VisualBertSportsClassifier(
  (visualbert): VisualBertModel(
    (embeddings): VisualBertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=1)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (visual_token_type_embeddings): Embedding(2, 768)
      (visual_position_embeddings): Embedding(512, 768)
      (visual_projection): Linear(in_features=2048, out_features=768, bias=True)
    )
    (encoder): VisualBertEncoder(
      (layer): ModuleList(
        (0-11): 12 x VisualBertLayer(
          (attention): VisualBertAttention(
            (self): VisualBertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
      

In [101]:
# =====================================================
# Forward de prueba (sin entrenar)
# =====================================================

batch = next(
    iter(train_loader)
)


input_ids = batch["input_ids"].to(
    DEVICE
)


attention_mask = batch["attention_mask"].to(
    DEVICE
)


visual_embeds = batch["visual_embeds"].to(
    DEVICE
)


with torch.no_grad():

    logits = model(
        input_ids,
        attention_mask,
        visual_embeds
    )


print(
    "Logits:",
    logits.shape
)

Logits: torch.Size([16, 4])


In [102]:
# =====================================================
# Configuración entrenamiento VisualBERT
# =====================================================

from pathlib import Path


# -----------------------------------------------------
# Hiperparámetros
# -----------------------------------------------------

if not hasattr(CONFIG, "learning_rate"):
    CONFIG.learning_rate = 1e-5


if not hasattr(CONFIG, "weight_decay"):
    CONFIG.weight_decay = 0.01


if not hasattr(CONFIG, "epochs"):
    CONFIG.epochs = 10



print("Learning rate:", CONFIG.learning_rate)
print("Weight decay :", CONFIG.weight_decay)
print("Epochs       :", CONFIG.epochs)



# -----------------------------------------------------
# Carpeta modelos
# -----------------------------------------------------

OUTPUT_MODELS = (
    ROOT /
    "outputs" /
    "models"
)


OUTPUT_MODELS.mkdir(
    parents=True,
    exist_ok=True
)


print("\nModelos se guardarán en:")
print(OUTPUT_MODELS)

Learning rate: 1e-05
Weight decay : 0.01
Epochs       : 10

Modelos se guardarán en:
/workspace/Exposicion3/notebook_MSR_VTT/outputs/models


In [103]:
#SOLO PARA ENTRENAR
# =====================================================
# Parte 3
# Training Loop VisualBERT
# =====================================================

import torch.optim as optim


# =====================================================
# Loss y Optimizer
# =====================================================

criterion = nn.CrossEntropyLoss()


optimizer = optim.AdamW(
    model.parameters(),
    lr=CONFIG.learning_rate,
    weight_decay=CONFIG.weight_decay
)



# =====================================================
# Función de entrenamiento
# =====================================================

def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device
):

    model.train()

    total_loss = 0
    correct = 0
    total = 0


    for batch in tqdm(
        loader,
        desc="Training"
    ):


        input_ids = batch["input_ids"].to(
            device
        )

        attention_mask = batch["attention_mask"].to(
            device
        )

        visual_embeds = batch["visual_embeds"].to(
            device
        )

        labels = batch["label"].to(
            device
        )


        # -----------------------------
        # Forward
        # -----------------------------

        optimizer.zero_grad()


        logits = model(
            input_ids,
            attention_mask,
            visual_embeds
        )


        loss = criterion(
            logits,
            labels
        )


        # -----------------------------
        # Backpropagation
        # -----------------------------

        loss.backward()


        optimizer.step()



        # -----------------------------
        # Estadísticas
        # -----------------------------

        total_loss += loss.item()


        predictions = torch.argmax(
            logits,
            dim=1
        )


        correct += (
            predictions == labels
        ).sum().item()


        total += labels.size(0)



    avg_loss = (
        total_loss /
        len(loader)
    )


    accuracy = (
        correct /
        total
    )


    return avg_loss, accuracy



# =====================================================
# Validación
# =====================================================

def evaluate(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0


    with torch.no_grad():

        for batch in tqdm(
            loader,
            desc="Validation"
        ):


            input_ids = batch["input_ids"].to(
                device
            )


            attention_mask = batch["attention_mask"].to(
                device
            )


            visual_embeds = batch["visual_embeds"].to(
                device
            )


            labels = batch["label"].to(
                device
            )


            logits = model(
                input_ids,
                attention_mask,
                visual_embeds
            )


            loss = criterion(
                logits,
                labels
            )


            total_loss += loss.item()


            predictions = torch.argmax(
                logits,
                dim=1
            )


            correct += (
                predictions == labels
            ).sum().item()


            total += labels.size(0)



    avg_loss = (
        total_loss /
        len(loader)
    )


    accuracy = (
        correct /
        total
    )


    return avg_loss, accuracy



# =====================================================
# Entrenamiento completo
# =====================================================

EPOCHS = CONFIG.epochs


best_val_acc = 0


for epoch in range(EPOCHS):


    print(
        f"\nEpoch {epoch+1}/{EPOCHS}"
    )


    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        DEVICE
    )


    val_loss, val_acc = evaluate(
        model,
        val_loader,
        criterion,
        DEVICE
    )


    print(
        f"Train Loss: {train_loss:.4f}"
    )

    print(
        f"Train Acc : {train_acc:.4f}"
    )

    print(
        f"Val Loss  : {val_loss:.4f}"
    )

    print(
        f"Val Acc   : {val_acc:.4f}"
    )


    # Guardar mejor modelo

    if val_acc > best_val_acc:

        best_val_acc = val_acc


        torch.save(
            model.state_dict(),
            OUTPUT_MODELS /
            "visualbert_sports_best.pt"
        )


        print(
            "Modelo guardado"
        )


print("\nEntrenamiento terminado")

print(
    "Mejor validación:",
    best_val_acc
)


Epoch 1/10


Validation: 100%|█████████████████████████████████| 2/2 [00:02<00:00,  1.07s/it]


Train Loss: 1.4572
Train Acc : 0.2308
Val Loss  : 1.5042
Val Acc   : 0.2963
Modelo guardado

Epoch 2/10


Validation: 100%|█████████████████████████████████| 2/2 [00:02<00:00,  1.34s/it]


Train Loss: 1.3955
Train Acc : 0.2692
Val Loss  : 1.4524
Val Acc   : 0.2963

Epoch 3/10


...

Train Loss: 0.5120
Train Acc : 0.9904
Val Loss  : 0.6501
Val Acc   : 0.8519

Epoch 9/10


Validation: 100%|█████████████████████████████████| 2/2 [00:01<00:00,  1.02it/s]


Train Loss: 0.3965
Train Acc : 0.9904
Val Loss  : 0.5888
Val Acc   : 0.8519

Epoch 10/10


Validation: 100%|█████████████████████████████████| 2/2 [00:02<00:00,  1.08s/it]


Train Loss: 0.3222
Train Acc : 0.9904
Val Loss  : 0.5173
Val Acc   : 0.8889
Modelo guardado

Entrenamiento terminado
Mejor validación: 0.8888888888888888


LXMERT
=============
La estructura actual:

{
 'video_id',
 'sport',
 'captions_blip',
 'num_frames',
 'visual_embeds',
 'input_ids',
 'attention_mask'
}

es suficiente para LXMERT.

La conversión será:
visual_embeds  ---> visual_feats de LXMERT;
input_ids      ---> texto;
attention_mask ---> máscara texto;
sport          ---> label;
video_id       ---> separación por video

In [104]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

for file in PROJECT_ROOT.rglob("*"):
    if file.suffix in [".pt", ".pth", ".bin"]:
        print(file)

/workspace/Exposicion3/notebook_MSR_VTT/data/processed/multimodal_train_context_12.pt
/workspace/Exposicion3/notebook_MSR_VTT/outputs/datasets/sports33_test_context_12_multimodal.pt
/workspace/Exposicion3/notebook_MSR_VTT/outputs/datasets/sports33_train_context_12_multimodal.pt
/workspace/Exposicion3/notebook_MSR_VTT/outputs/models/visualbert_sports_best.pt


In [105]:
from pathlib import Path
import torch


DATASET_FILE = Path(
    "/workspace/Exposicion3/notebook_MSR_VTT/outputs/datasets/sports33_train_context_12_multimodal.pt"
)


data = torch.load(
    DATASET_FILE,
    map_location="cpu"
)


print(type(data))


if isinstance(data, dict):
    print(data.keys())


elif isinstance(data, list):
    print("Número de muestras:", len(data))
    print(data[0].keys())

/tmp/ipykernel_25/3377954786.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(


<class 'list'>
Número de muestras: 131
dict_keys(['video_id', 'sport', 'captions_blip', 'num_frames', 'visual_embeds', 'input_ids', 'attention_mask'])


In [106]:
# =====================================================
# 04_build_lxmert_dataset.py
# Convert existing multimodal dataset to LXMERT format
# =====================================================

from pathlib import Path
import torch
import random


# -----------------------------------------------------
# Paths
# -----------------------------------------------------

DATASET_DIR = Path(
    "/workspace/Exposicion3/notebook_MSR_VTT/outputs/datasets"
)

OUTPUT_DIR = Path(
    "/workspace/Exposicion3/notebook_MSR_VTT/outputs/lxmert_dataset"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


TRAIN_FILE = (
    DATASET_DIR /
    "sports33_train_context_12_multimodal.pt"
)

TEST_FILE = (
    DATASET_DIR /
    "sports33_test_context_12_multimodal.pt"
)


# -----------------------------------------------------
# Load
# -----------------------------------------------------

train_data = torch.load(
    TRAIN_FILE,
    map_location="cpu"
)

test_data = torch.load(
    TEST_FILE,
    map_location="cpu"
)


print(
    "Train:",
    len(train_data)
)

print(
    "Test:",
    len(test_data)
)


# -----------------------------------------------------
# Convert format
# -----------------------------------------------------

def convert_to_lxmert(samples):

    lxmert_samples = []

    for item in samples:

        sample = {

            # identificadores
            "video_id":
                item["video_id"],

            "sport":
                item["sport"],


            # visual input
            #
            # expected:
            # [frames, feature_dim]
            #
            "visual_feats":
                item["visual_embeds"],


            # text input
            "input_ids":
                item["input_ids"],

            "attention_mask":
                item["attention_mask"],


            # temporal information
            "num_frames":
                item["num_frames"]
        }


        lxmert_samples.append(sample)


    return lxmert_samples



train_lxmert = convert_to_lxmert(
    train_data
)

test_lxmert = convert_to_lxmert(
    test_data
)


print(
    train_lxmert[0].keys()
)


print(
    train_lxmert[0]["visual_feats"].shape
)


# -----------------------------------------------------
# Save
# -----------------------------------------------------

torch.save(
    train_lxmert,
    OUTPUT_DIR /
    "lxmert_train_context_12.pt"
)


torch.save(
    test_lxmert,
    OUTPUT_DIR /
    "lxmert_test_context_12.pt"
)


print("Guardado correctamente")

/tmp/ipykernel_25/4134506579.py:44: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  train_data = torch.load(
/tmp/ipykernel_25/4134506579.py:49: FutureWarning: You are using `

Train: 131
Test: 18
dict_keys(['video_id', 'sport', 'visual_feats', 'input_ids', 'attention_mask', 'num_frames'])
torch.Size([12, 2048])
Guardado correctamente


siguiente paso: adaptar el modelo LXMERT de HuggingFace, LXMERT originalmente espera visual_feats + visual_pos (regiones de objetos). Como nosotros tenemos tokens temporales de frames [12,2048].


visual_embeds (12,2048)
          |
          ↓
Linear(2048 → 768)
          |
          ↓
LXMERT visual input
          |
          ↓
LXMERT encoder
          |
          ↓
clasificador deporte

# =====================================================
# 05_train_lxmert.py
# LXMERT adapted for temporal visual tokens
# =====================================================

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from transformers import (
    LxmertModel,
    LxmertConfig
)


# =====================================================
# Device
# =====================================================

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)



# =====================================================
# Dataset
# =====================================================

class LXMERTDataset(Dataset):

    def __init__(self, file):

        self.data = torch.load(
            file,
            map_location="cpu"
        )

        # map sports to ids
        sports = sorted(
            list(
                set(
                    x["sport"]
                    for x in self.data
                )
            )
        )

        self.sport2id = {
            s:i for i,s in enumerate(sports)
        }


    def __len__(self):

        return len(self.data)


    def __getitem__(self, idx):

        item = self.data[idx]


        return {

            "visual_feats":
                item["visual_feats"].float(),

            "input_ids":
                item["input_ids"],

            "attention_mask":
                item["attention_mask"],

            "label":
                torch.tensor(
                    self.sport2id[item["sport"]],
                    dtype=torch.long
                )
        }



# =====================================================
# Files
# =====================================================

TRAIN_FILE = (
    "/workspace/Exposicion3/notebook_MSR_VTT/"
    "outputs/lxmert_dataset/"
    "lxmert_train_context_12.pt"
)


TEST_FILE = (
    "/workspace/Exposicion3/notebook_MSR_VTT/"
    "outputs/lxmert_dataset/"
    "lxmert_test_context_12.pt"
)


train_dataset = LXMERTDataset(
    TRAIN_FILE
)


test_dataset = LXMERTDataset(
    TEST_FILE
)


train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)


# =====================================================
# Check batch
# =====================================================

batch = next(
    iter(train_loader)
)


for k,v in batch.items():

    print(
        k,
        v.shape
    )

In [39]:
# =====================================================
# Temporal LXMERT Classifier
# =====================================================
import torch
import torch.nn as nn

from transformers import LxmertModel


class TemporalLXMERTClassifier(nn.Module):

    def __init__(
        self,
        num_classes
    ):
        super().__init__()


        self.lxmert = LxmertModel.from_pretrained(
            "unc-nlp/lxmert-base-uncased"
        )


        self.classifier = nn.Linear(
            768,
            num_classes
        )



    def forward(
        self,
        visual_feats,
        input_ids,
        attention_mask
    ):


        batch_size = visual_feats.size(0)


        # LXMERT espera posiciones visuales
        visual_pos = torch.zeros(
            batch_size,
            visual_feats.size(1),
            4,
            device=visual_feats.device
        )


        outputs = self.lxmert(
            input_ids=input_ids,
            attention_mask=attention_mask,

            visual_feats=visual_feats,
            visual_pos=visual_pos
        )


        pooled = outputs.pooled_output


        logits = self.classifier(
            pooled
        )


        return logits

In [115]:
#hacer un forward pass, ejecutar con un solo batch de datos por el modelo sin entrenar todavía.
num_classes = len(train_dataset.sport2id)

model = TemporalLXMERTClassifier(
    num_classes
).to(device)


batch = next(iter(train_loader))


batch = {
    k:v.to(device)
    for k,v in batch.items()
}


logits = model(
    batch["visual_feats"],
    batch["input_ids"],
    batch["attention_mask"]
)


print(logits.shape)

Some weights of the model checkpoint at unc-nlp/lxmert-base-uncased were not used when initializing LxmertModel: ['answer_head.logit_fc.0.bias', 'answer_head.logit_fc.0.weight', 'answer_head.logit_fc.2.bias', 'answer_head.logit_fc.2.weight', 'answer_head.logit_fc.3.bias', 'answer_head.logit_fc.3.weight', 'cls.predictions.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight', 'obj_predict_head.decoder_dict.attr.bias', 'obj_predict_head.decoder_dict.attr.weight', 'obj_predict_head.decoder_dict.feat.bias', 'obj_predict_head.decoder_dict.feat.weight', 'obj_predict_head.decoder_dict.obj.bias', 'obj_predict_head.decoder_dict.obj.weight', 'obj_predict_head.transform.LayerNorm.bias', 'obj_predict_head.transform.LayerNorm.weight', 'obj_predict_head.transform.dense.bias', 'obj_pred

torch.Size([8, 4])


In [116]:
print(train_dataset.sport2id)

{'basketball': 0, 'soccer': 1, 'swimming': 2, 'tennis': 3}


In [117]:
# =====================================================
# 06_train_lxmert.py
# Train Temporal LXMERT
# =====================================================

import torch
import torch.nn as nn

from torch.optim import AdamW
from sklearn.metrics import accuracy_score


# -----------------------------------------------------
# Model
# -----------------------------------------------------

num_classes = len(
    train_dataset.sport2id
)

model = TemporalLXMERTClassifier(
    num_classes
).to(device)



# -----------------------------------------------------
# Loss and optimizer
# -----------------------------------------------------

criterion = nn.CrossEntropyLoss()


optimizer = AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)



# -----------------------------------------------------
# Training config
# -----------------------------------------------------

EPOCHS = 10


best_acc = 0.0


MODEL_PATH = (
    "/workspace/Exposicion3/notebook_MSR_VTT/"
    "outputs/models/"
    "lxmert_sports_best.pt"
)



# -----------------------------------------------------
# Training loop
# -----------------------------------------------------

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0
    predictions = []
    targets = []


    for batch in train_loader:


        batch = {
            k:v.to(device)
            for k,v in batch.items()
        }


        optimizer.zero_grad()


        logits = model(
            batch["visual_feats"],
            batch["input_ids"],
            batch["attention_mask"]
        )


        loss = criterion(
            logits,
            batch["label"]
        )


        loss.backward()

        optimizer.step()


        total_loss += loss.item()


        preds = torch.argmax(
            logits,
            dim=1
        )


        predictions.extend(
            preds.cpu().numpy()
        )

        targets.extend(
            batch["label"].cpu().numpy()
        )


    acc = accuracy_score(
        targets,
        predictions
    )


    avg_loss = (
        total_loss /
        len(train_loader)
    )


    print(
        f"Epoch {epoch+1}/{EPOCHS} "
        f"Loss={avg_loss:.4f} "
        f"Acc={acc:.4f}"
    )


    # guardar mejor modelo

    if acc > best_acc:

        best_acc = acc

        torch.save(
            {
                "model_state_dict":
                    model.state_dict(),

                "sport2id":
                    train_dataset.sport2id
            },
            MODEL_PATH
        )

        print(
            "Modelo guardado"
        )

/usr/local/lib/python3.11/site-packages/sklearn/utils/_param_validation.py:11: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.4.6)
  from scipy.sparse import csr_matrix, issparse
Some weights of the model checkpoint at unc-nlp/lxmert-base-uncased were not used when initializing LxmertModel: ['answer_head.logit_fc.0.bias', 'answer_head.logit_fc.0.weight', 'answer_head.logit_fc.2.bias', 'answer_head.logit_fc.2.weight', 'answer_head.logit_fc.3.bias', 'answer_head.logit_fc.3.weight', 'cls.predictions.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight', 'obj_predict_head.decoder_dict.attr.bias', 'obj_predict_head.decoder_dict.attr.weight', 'obj_predict_head.decoder_dict.feat.bias', 'obj_predict_head.decoder_dict.fea

Epoch 1/10 Loss=1.1347 Acc=0.5954
Modelo guardado
Epoch 2/10 Loss=0.5653 Acc=0.8931
Modelo guardado
Epoch 3/10 Loss=0.2770 Acc=0.9313
Modelo guardado
Epoch 4/10 Loss=0.1389 Acc=0.9771
Modelo guardado
Epoch 5/10 Loss=0.0900 Acc=0.9847
Modelo guardado
Epoch 6/10 Loss=0.0561 Acc=0.9924
Modelo guardado
Epoch 7/10 Loss=0.0337 Acc=1.0000
Modelo guardado
Epoch 8/10 Loss=0.0193 Acc=1.0000
Epoch 9/10 Loss=0.0139 Acc=1.0000
Epoch 10/10 Loss=0.0111 Acc=1.0000


EVALUACION DE LXMERT CON MUESTRA TEST
===================================

In [2]:
# =====================================================
# 07_evaluate_lxmert.py
# Evaluate Temporal LXMERT
# =====================================================

import torch
import torch.nn as nn

from torch.utils.data import DataLoader

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [40]:
#Definir Dataset (común)
class LXMERTDataset(Dataset):

    def __init__(self, file):

        self.data = torch.load(
            file,
            map_location="cpu"
        )

        sports = sorted(
            list(
                set(
                    x["sport"]
                    for x in self.data
                )
            )
        )

        self.sport2id = {
            s:i for i,s in enumerate(sports)
        }


    def __len__(self):

        return len(self.data)


    def __getitem__(self, idx):

        item = self.data[idx]

        return {

            "visual_feats":
                item["visual_feats"].float(),

            "input_ids":
                item["input_ids"],

            "attention_mask":
                item["attention_mask"],

            "label":
                torch.tensor(
                    self.sport2id[item["sport"]],
                    dtype=torch.long
                )
        }

# =====================================================
# Temporal LXMERT Classifier
# =====================================================

import torch
import torch.nn as nn

from transformers import LxmertModel


class TemporalLXMERTClassifier(nn.Module):

    def __init__(self, num_classes):

        super().__init__()

        self.lxmert = LxmertModel.from_pretrained(
            "unc-nlp/lxmert-base-uncased"
        )


        self.classifier = nn.Linear(
            768,
            num_classes
        )


    def forward(
        self,
        visual_feats,
        input_ids,
        attention_mask
    ):

        batch_size = visual_feats.size(0)


        visual_pos = torch.zeros(
            batch_size,
            visual_feats.size(1),
            4,
            device=visual_feats.device
        )


        outputs = self.lxmert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            visual_feats=visual_feats,
            visual_pos=visual_pos
        )


        pooled = outputs.pooled_output


        logits = self.classifier(
            pooled
        )


        return logits

In [11]:
#Dataset de prueba
TEST_FILE = (
    "/workspace/Exposicion3/notebook_MSR_VTT/"
    "outputs/lxmert_dataset/"
    "lxmert_test_context_12.pt"
)

test_dataset = LXMERTDataset(TEST_FILE)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False
)

/tmp/ipykernel_38/3245947059.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data = torch.load(


In [41]:
#CARGAR EL MODELO PREENTRENADO
MODEL_PATH = (
    "/workspace/Exposicion3/notebook_MSR_VTT/"
    "outputs/models/"
    "lxmert_sports_best.pt"
)

checkpoint = torch.load(
    MODEL_PATH,
    map_location=device
)

num_classes = len(checkpoint["sport2id"])

model = TemporalLXMERTClassifier(
    num_classes
).to(device)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("Modelo cargado correctamente.")

/tmp/ipykernel_38/3465255387.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(
Some weights of the model checkpoint at unc-nlp/lxmert-base-uncased

Modelo cargado correctamente.


In [13]:
print(checkpoint["sport2id"])

{'basketball': 0, 'soccer': 1, 'swimming': 2, 'tennis': 3}


In [14]:
TEST_FILE = (
    "/workspace/Exposicion3/notebook_MSR_VTT/"
    "outputs/lxmert_dataset/"
    "lxmert_test_context_12.pt"
)

test_dataset = LXMERTDataset(TEST_FILE)

# usar el mismo mapeo del entrenamiento
test_dataset.sport2id = checkpoint["sport2id"]


test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False
)

print("Número de muestras:", len(test_dataset))
print("Número de batches:", len(test_loader))

Número de muestras: 18
Número de batches: 3


/tmp/ipykernel_38/3245947059.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data = torch.load(


In [15]:
from tqdm.auto import tqdm

predictions = []
targets = []

model.eval()

with torch.no_grad():

    for batch in tqdm(test_loader):

        batch = {
            k:v.to(device)
            for k,v in batch.items()
        }


        logits = model(
            batch["visual_feats"],
            batch["input_ids"],
            batch["attention_mask"]
        )


        preds = torch.argmax(
            logits,
            dim=1
        )


        predictions.extend(
            preds.cpu().numpy()
        )

        targets.extend(
            batch["label"].cpu().numpy()
        )


print("Evaluación terminada")

  0%|          | 0/3 [00:00<?, ?it/s]

Evaluación terminada


In [16]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

accuracy = accuracy_score(
    targets,
    predictions
)

print("="*60)
print("LXMERT TEST RESULTS")
print("="*60)

print(f"\nAccuracy: {accuracy:.4f}")


print("\nClassification Report:\n")

print(
    classification_report(
        targets,
        predictions,
        target_names=[
            "basketball",
            "soccer",
            "swimming",
            "tennis"
        ],
        digits=4
    )
)


print("\nConfusion Matrix:\n")

cm = confusion_matrix(
    targets,
    predictions
)

print(cm)

LXMERT TEST RESULTS

Accuracy: 0.8889

Classification Report:

              precision    recall  f1-score   support

  basketball     1.0000    0.6000    0.7500         5
      soccer     0.8333    1.0000    0.9091         5
    swimming     0.7500    1.0000    0.8571         3
      tennis     1.0000    1.0000    1.0000         5

    accuracy                         0.8889        18
   macro avg     0.8958    0.9000    0.8791        18
weighted avg     0.9120    0.8889    0.8815        18


Confusion Matrix:

[[3 1 1 0]
 [0 5 0 0]
 [0 0 3 0]
 [0 0 0 5]]


##INTERPRETACION
"El modelo LXMERT alcanzó una exactitud del 88.89% sobre el conjunto de prueba. El desempeño fue especialmente alto en tenis (F1=1.00), soccer (F1=0.91) y swimming (F1=0.86). La principal fuente de error estuvo en la clase basketball, donde algunas muestras fueron confundidas con otros deportes, indicando que ciertas configuraciones visuales comparten características similares entre disciplinas."

Tennis: perfecto.
5/5 correctamente clasificados.
Soccer: perfecto en recall.
Detectó todos los casos de soccer.
Swimming:
También detectó todos los casos.
Basketball:
Es donde falla.
De 5 muestras:
3 correctas.
1 confundida con soccer.
1 confundida con swimming.

Train accuracy: 100%
Test accuracy: 88.89%

EVALUACION de VISUALBERT CON MUESTRA TEST
========================================
Ejecutar el paso 1 de reproducibilidad de visualbert
Ejecutar paso 2, pero no paso 3 de entrenamiento.

In [27]:
# =====================================================
# Cargar modelo VisualBERT entrenado
# =====================================================
# -----------------------------------------------------
# Carpeta modelos
# -----------------------------------------------------

OUTPUT_MODELS = (
    ROOT /
    "outputs" /
    "models"
)


OUTPUT_MODELS.mkdir(
    parents=True,
    exist_ok=True
)


print("\nModelos se guardarán en:")
print(OUTPUT_MODELS)
model = VisualBertSportsClassifier(
    visualbert,
    NUM_CLASSES
).to(DEVICE)


MODEL_PATH = (
    OUTPUT_MODELS /
    "visualbert_sports_best.pt"
)


state_dict = torch.load(
    MODEL_PATH,
    map_location=DEVICE
)


model.load_state_dict(
    state_dict
)


model.eval()


print("VisualBERT cargado correctamente")


Modelos se guardarán en:
/workspace/Exposicion3/notebook_MSR_VTT/outputs/models


/tmp/ipykernel_38/4233915067.py:35: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(


VisualBERT cargado correctamente


In [28]:
# =====================================================
# Dataset Test VisualBERT
# =====================================================

test_path = (
    OUTPUT_DATASETS /
    f"sports33_test_{CONFIG.frame_context}_multimodal.pt"
)


test_data = torch.load(
    test_path,
    weights_only=False
)


test_dataset = SportsDataset(
    test_data
)


test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG.batch_size,
    shuffle=False
)


print(
    "Número de muestras test:",
    len(test_dataset)
)

Número de muestras test: 18


In [29]:
# =====================================================
# Inferencia VisualBERT en test
# =====================================================

from tqdm.auto import tqdm

predictions = []
targets = []


model.eval()


with torch.no_grad():

    for batch in tqdm(test_loader):

        input_ids = batch["input_ids"].to(DEVICE)

        attention_mask = batch["attention_mask"].to(DEVICE)

        visual_embeds = batch["visual_embeds"].to(DEVICE)

        labels = batch["label"].to(DEVICE)


        logits = model(
            input_ids,
            attention_mask,
            visual_embeds
        )


        preds = torch.argmax(
            logits,
            dim=1
        )


        predictions.extend(
            preds.cpu().numpy()
        )

        targets.extend(
            labels.cpu().numpy()
        )


print("Inferencia terminada")

  0%|          | 0/2 [00:00<?, ?it/s]

Inferencia terminada


In [30]:
# =====================================================
# Métricas VisualBERT
# =====================================================

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)


accuracy = accuracy_score(
    targets,
    predictions
)


print("="*60)
print("VISUALBERT TEST RESULTS")
print("="*60)

print(
    f"\nAccuracy: {accuracy:.4f}"
)


print(
    "\nClassification Report:\n"
)


print(
    classification_report(
        targets,
        predictions,
        target_names=[
            "basketball",
            "soccer",
            "swimming",
            "tennis"
        ],
        digits=4
    )
)


print(
    "\nConfusion Matrix:\n"
)


print(
    confusion_matrix(
        targets,
        predictions
    )
)

VISUALBERT TEST RESULTS

Accuracy: 0.7778

Classification Report:

              precision    recall  f1-score   support

  basketball     0.8000    0.8000    0.8000         5
      soccer     1.0000    0.8000    0.8889         5
    swimming     0.6667    0.6667    0.6667         3
      tennis     0.6667    0.8000    0.7273         5

    accuracy                         0.7778        18
   macro avg     0.7833    0.7667    0.7707        18
weighted avg     0.7963    0.7778    0.7823        18


Confusion Matrix:

[[4 0 0 1]
 [1 4 0 0]
 [0 0 2 1]
 [0 0 1 4]]


##Interpretación:

Basketball
4/5 correctos.
1 confundido con tennis.
Soccer
4/5 correctos.
1 confundido con basketball.
Swimming
2/3 correctos.
1 confundido con tennis.
Tennis
4/5 correctos.
1 confundido con swimming.

Comparación interesante

LXMERT:

Accuracy = 88.89%

VisualBERT:

Accuracy = 77.78%

Diferencia:

88.89 - 77.78 = 11.11 puntos porcentuales

PERTURBACIONES
==============

Empezaremos con Shuffle, que es la perturbación más representativa.

In [32]:
# =====================================================
# Crear dataset TEST con Shuffle Temporal
# =====================================================

import copy
import torch
from pathlib import Path


INPUT_FILE = (
    OUTPUT_DATASETS /
    f"sports33_test_{CONFIG.frame_context}_multimodal.pt"
)

OUTPUT_FILE = (
    OUTPUT_DATASETS /
    f"sports33_test_{CONFIG.frame_context}_shuffle.pt"
)


data = torch.load(
    INPUT_FILE,
    weights_only=False
)

shuffle_data = []


g = torch.Generator().manual_seed(CONFIG.seed)


for sample in data:

    new_sample = copy.deepcopy(sample)

    visual = new_sample["visual_embeds"]

    perm = torch.randperm(
        visual.shape[0],
        generator=g
    )

    new_sample["visual_embeds"] = visual[perm]

    shuffle_data.append(new_sample)


torch.save(
    shuffle_data,
    OUTPUT_FILE
)

print("Guardado en:")
print(OUTPUT_FILE)

print("Número de muestras:", len(shuffle_data))

Guardado en:
/workspace/Exposicion3/notebook_MSR_VTT/outputs/datasets/sports33_test_context_12_shuffle.pt
Número de muestras: 18


In [33]:
#CARGAR EL DATASET SUFFLE
shuffle_data = torch.load(
    OUTPUT_FILE,
    weights_only=False
)

shuffle_dataset = SportsDataset(
    shuffle_data
)

shuffle_loader = DataLoader(
    shuffle_dataset,
    batch_size=CONFIG.batch_size,
    shuffle=False
)

print("Número de muestras:", len(shuffle_dataset))

Número de muestras: 18


In [34]:
predictions = []
targets = []

model.eval()

with torch.no_grad():

    for batch in tqdm(shuffle_loader):

        input_ids = batch["input_ids"].to(DEVICE)

        attention_mask = batch["attention_mask"].to(DEVICE)

        visual_embeds = batch["visual_embeds"].to(DEVICE)

        labels = batch["label"].to(DEVICE)

        logits = model(
            input_ids,
            attention_mask,
            visual_embeds
        )

        preds = torch.argmax(
            logits,
            dim=1
        )

        predictions.extend(
            preds.cpu().numpy()
        )

        targets.extend(
            labels.cpu().numpy())

print("Inferencia terminada")

  0%|          | 0/2 [00:00<?, ?it/s]

Inferencia terminada


In [35]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

print("=" * 60)
print("VISUALBERT + SHUFFLE")
print("=" * 60)

print(f"Accuracy: {accuracy_score(targets, predictions):.4f}")

print("\nClassification Report:\n")

print(
    classification_report(
        targets,
        predictions,
        target_names=[
            "basketball",
            "soccer",
            "swimming",
            "tennis"
        ],
        digits=4
    )
)

print("\nConfusion Matrix:\n")

print(confusion_matrix(targets, predictions))

VISUALBERT + SHUFFLE
Accuracy: 0.7778

Classification Report:

              precision    recall  f1-score   support

  basketball     0.8000    0.8000    0.8000         5
      soccer     1.0000    0.8000    0.8889         5
    swimming     0.6667    0.6667    0.6667         3
      tennis     0.6667    0.8000    0.7273         5

    accuracy                         0.7778        18
   macro avg     0.7833    0.7667    0.7707        18
weighted avg     0.7963    0.7778    0.7823        18


Confusion Matrix:

[[4 0 0 1]
 [1 4 0 0]
 [0 0 2 1]
 [0 0 1 4]]


In [36]:
original = torch.load(
    INPUT_FILE,
    weights_only=False
)

shuffle = torch.load(
    OUTPUT_FILE,
    weights_only=False
)

print(torch.equal(
    original[0]["visual_embeds"],
    shuffle[0]["visual_embeds"]
))

False


Lxmert suffle
=================

In [37]:
# =====================================================
# Crear dataset LXMERT con Shuffle
# =====================================================

import copy
import torch

INPUT_FILE = (
    "/workspace/Exposicion3/notebook_MSR_VTT/"
    "outputs/lxmert_dataset/"
    "lxmert_test_context_12.pt"
)

OUTPUT_FILE = (
    "/workspace/Exposicion3/notebook_MSR_VTT/"
    "outputs/lxmert_dataset/"
    "lxmert_test_context_12_shuffle.pt"
)


data = torch.load(
    INPUT_FILE,
    map_location="cpu"
)

shuffle_data = []

g = torch.Generator().manual_seed(CONFIG.seed)

for sample in data:

    new_sample = copy.deepcopy(sample)

    visual = new_sample["visual_feats"]

    perm = torch.randperm(
        visual.shape[0],
        generator=g
    )

    new_sample["visual_feats"] = visual[perm]

    shuffle_data.append(new_sample)


torch.save(
    shuffle_data,
    OUTPUT_FILE
)

print("Guardado correctamente")
print(OUTPUT_FILE)
print("Número de muestras:", len(shuffle_data))

/tmp/ipykernel_38/979866187.py:21: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(


Guardado correctamente
/workspace/Exposicion3/notebook_MSR_VTT/outputs/lxmert_dataset/lxmert_test_context_12_shuffle.pt
Número de muestras: 18


In [38]:
#cargar
shuffle_dataset = LXMERTDataset(
    OUTPUT_FILE
)

shuffle_loader = DataLoader(
    shuffle_dataset,
    batch_size=8,
    shuffle=False
)

print(len(shuffle_dataset))

18


/tmp/ipykernel_38/3245947059.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data = torch.load(


In [ ]:
ejecutar la celda: 
# =====================================================
# Temporal LXMERT Classifier
# =====================================================
Ejecutar la celda: 
#CARGAR EL MODELO PREENTRENADO


In [42]:
from tqdm import tqdm

predictions = []
targets = []

model.eval()

with torch.no_grad():

    for batch in tqdm(shuffle_loader):

        batch = {
            k: v.to(device)
            for k, v in batch.items()
        }

        logits = model(
            batch["visual_feats"],
            batch["input_ids"],
            batch["attention_mask"]
        )

        preds = torch.argmax(
            logits,
            dim=1
        )

        predictions.extend(
            preds.cpu().numpy()
        )

        targets.extend(
            batch["label"].cpu().numpy()
        )

print("Inferencia terminada")

100%|█████████████████████████████████████████████| 3/3 [00:01<00:00,  1.92it/s]

Inferencia terminada


In [43]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

print("=" * 60)
print("LXMERT + SHUFFLE")
print("=" * 60)

print(f"Accuracy: {accuracy_score(targets, predictions):.4f}")

print("\nClassification Report:\n")

print(
    classification_report(
        targets,
        predictions,
        target_names=[
            "basketball",
            "soccer",
            "swimming",
            "tennis"
        ],
        digits=4
    )
)

print("\nConfusion Matrix:\n")

print(
    confusion_matrix(
        targets,
        predictions
    )
)

LXMERT + SHUFFLE
Accuracy: 0.8889

Classification Report:

              precision    recall  f1-score   support

  basketball     1.0000    0.6000    0.7500         5
      soccer     0.8333    1.0000    0.9091         5
    swimming     0.7500    1.0000    0.8571         3
      tennis     1.0000    1.0000    1.0000         5

    accuracy                         0.8889        18
   macro avg     0.8958    0.9000    0.8791        18
weighted avg     0.9120    0.8889    0.8815        18


Confusion Matrix:

[[3 1 1 0]
 [0 5 0 0]
 [0 0 3 0]
 [0 0 0 5]]


Crear data Muestra -6 frames

In [44]:
# =====================================================
# Crear dataset TEST Sampling (12 -> 6 frames)
# =====================================================

import copy
import torch

INPUT_FILE = (
    OUTPUT_DATASETS /
    f"sports33_test_{CONFIG.frame_context}_multimodal.pt"
)

OUTPUT_FILE = (
    OUTPUT_DATASETS /
    f"sports33_test_{CONFIG.frame_context}_sampling6.pt"
)


data = torch.load(
    INPUT_FILE,
    weights_only=False
)

sampling_data = []


# índices seleccionados
selected_idx = [0, 2, 4, 6, 8, 10]


for sample in data:

    new_sample = copy.deepcopy(sample)


    # -------------------------------------
    # Visual Embeddings
    # -------------------------------------

    new_sample["visual_embeds"] = (
        new_sample["visual_embeds"][selected_idx]
    )


    # -------------------------------------
    # BLIP captions
    # -------------------------------------

    if isinstance(
        new_sample["captions_blip"],
        list
    ):

        new_sample["captions_blip"] = [

            new_sample["captions_blip"][i]

            for i in selected_idx
        ]


    # -------------------------------------
    # actualizar número de frames
    # -------------------------------------

    new_sample["num_frames"] = len(
        selected_idx
    )


    sampling_data.append(
        new_sample
    )


torch.save(
    sampling_data,
    OUTPUT_FILE
)

print("Guardado en:")
print(OUTPUT_FILE)

print("Número de muestras:", len(sampling_data))

Guardado en:
/workspace/Exposicion3/notebook_MSR_VTT/outputs/datasets/sports33_test_context_12_sampling6.pt
Número de muestras: 18


In [45]:
sample = sampling_data[0]

print(sample["visual_embeds"].shape)

print(sample["num_frames"])

print(len(sample["captions_blip"]))

print(sample["captions_blip"])

torch.Size([6, 2048])
6
6
['a tennis court with a green court and white lines', 'a person is pointing at a tennis court', 'a person is pointing at a tennis court', 'a person is pointing at a tennis court', 'a person is pointing at a tennis court', 'a person is pointing at a tennis court']


Cargar el nuevo dataset Frames 6
==

In [46]:
sampling_data = torch.load(
    OUTPUT_FILE,
    weights_only=False
)

sampling_dataset = SportsDataset(
    sampling_data
)

sampling_loader = DataLoader(
    sampling_dataset,
    batch_size=CONFIG.batch_size,
    shuffle=False
)

print("Número de muestras:", len(sampling_dataset))

Número de muestras: 18


In [50]:
#Verificación 
sample = sampling_dataset[0]

print(sample["visual_embeds"].dtype)
print(sample["input_ids"].dtype)
print(sample["attention_mask"].dtype)

torch.float32
torch.int64
torch.int64


In [52]:
batch = next(iter(sampling_loader))

print(batch["visual_embeds"].dtype)
print(batch["input_ids"].dtype)
print(batch["attention_mask"].dtype)
print(type(model))

torch.float32
torch.int64
torch.int64
<class '__main__.TemporalLXMERTClassifier'>


# lxmert con Sampling6

In [53]:
# =====================================================
# Crear dataset LXMERT Sampling-6
# =====================================================

import copy
import torch

INPUT_FILE = (
    "/workspace/Exposicion3/notebook_MSR_VTT/"
    "outputs/lxmert_dataset/"
    "lxmert_test_context_12.pt"
)

OUTPUT_FILE = (
    "/workspace/Exposicion3/notebook_MSR_VTT/"
    "outputs/lxmert_dataset/"
    "lxmert_test_context_12_sampling6.pt"
)


data = torch.load(
    INPUT_FILE,
    map_location="cpu"
)

sampling_data = []

# Mantener la misma semilla del proyecto
g = torch.Generator().manual_seed(CONFIG.seed)

# Seleccionar 6 de los 12 frames
selected_idx = [0, 2, 4, 6, 8, 10]

for sample in data:

    new_sample = copy.deepcopy(sample)

    new_sample["visual_feats"] = (
        new_sample["visual_feats"][selected_idx].float()
    )

    new_sample["num_frames"] = len(selected_idx)

    sampling_data.append(new_sample)


torch.save(
    sampling_data,
    OUTPUT_FILE
)

print("Guardado correctamente")
print(OUTPUT_FILE)
print("Número de muestras:", len(sampling_data))

Guardado correctamente
/workspace/Exposicion3/notebook_MSR_VTT/outputs/lxmert_dataset/lxmert_test_context_12_sampling6.pt
Número de muestras: 18


/tmp/ipykernel_38/231333944.py:21: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(


In [54]:
#carga el dataset con frames 6
sampling_dataset = LXMERTDataset(
    OUTPUT_FILE
)

sampling_loader = DataLoader(
    sampling_dataset,
    batch_size=8,
    shuffle=False
)

print(len(sampling_dataset))

18


/tmp/ipykernel_38/3245947059.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data = torch.load(


In [55]:
#validación de LXMert acepta los números de variables de caracteristicas visuales
batch = next(iter(sampling_loader))

for k, v in batch.items():
    print(k, v.shape, v.dtype)

visual_feats torch.Size([8, 6, 2048]) torch.float32
input_ids torch.Size([8, 40]) torch.int64
attention_mask torch.Size([8, 40]) torch.int64
label torch.Size([8]) torch.int64


In [56]:
from tqdm import tqdm

predictions = []
targets = []

model.eval()

with torch.no_grad():

    for batch in tqdm(sampling_loader):

        batch = {
            k: v.to(device)
            for k, v in batch.items()
        }

        logits = model(
            batch["visual_feats"],
            batch["input_ids"],
            batch["attention_mask"]
        )

        preds = torch.argmax(
            logits,
            dim=1
        )

        predictions.extend(
            preds.cpu().numpy()
        )

        targets.extend(
            batch["label"].cpu().numpy()
        )

print("Inferencia terminada")

100%|█████████████████████████████████████████████| 3/3 [00:02<00:00,  1.12it/s]

Inferencia terminada


In [57]:
#Celda 5 - Resultados
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

print("=" * 60)
print("LXMERT + SAMPLING (6 frames)")
print("=" * 60)

print(f"Accuracy: {accuracy_score(targets, predictions):.4f}")

print("\nClassification Report:\n")

print(
    classification_report(
        targets,
        predictions,
        target_names=[
            "basketball",
            "soccer",
            "swimming",
            "tennis"
        ],
        digits=4
    )
)

print("\nConfusion Matrix:\n")

print(
    confusion_matrix(
        targets,
        predictions
    )
)

LXMERT + SAMPLING (6 frames)
Accuracy: 0.8889

Classification Report:

              precision    recall  f1-score   support

  basketball     1.0000    0.6000    0.7500         5
      soccer     0.8333    1.0000    0.9091         5
    swimming     0.7500    1.0000    0.8571         3
      tennis     1.0000    1.0000    1.0000         5

    accuracy                         0.8889        18
   macro avg     0.8958    0.9000    0.8791        18
weighted avg     0.9120    0.8889    0.8815        18


Confusion Matrix:

[[3 1 1 0]
 [0 5 0 0]
 [0 0 3 0]
 [0 0 0 5]]


# VisualBert con Sampling6

EVALUACION de VISUALBERT CON MUESTRA SAMPLING6
========================================
Ejecutar el paso 1 de reproducibilidad de visualbert
 Configuración entrenamiento VisualBERT

Ejecutar paso 2, pero no paso 3 de entrenamiento.

In [62]:
sampling_data = torch.load(
    OUTPUT_DATASETS /
    f"sports33_test_{CONFIG.frame_context}_sampling6.pt",
    weights_only=False
)

sampling_dataset = SportsDataset(
    sampling_data
)

sampling_loader = DataLoader(
    sampling_dataset,
    batch_size=CONFIG.batch_size,
    shuffle=False
)

print(len(sampling_dataset))

18


In [63]:
print(type(model))

<class '__main__.VisualBertSportsClassifier'>


In [64]:
from tqdm import tqdm

predictions = []
targets = []

model.eval()

with torch.no_grad():

    for batch in tqdm(sampling_loader):

        input_ids = batch["input_ids"].to(DEVICE)

        attention_mask = batch["attention_mask"].to(DEVICE)

        visual_embeds = batch["visual_embeds"].to(DEVICE)

        labels = batch["label"].to(DEVICE)

        logits = model(
            input_ids,
            attention_mask,
            visual_embeds
        )

        preds = torch.argmax(
            logits,
            dim=1
        )

        predictions.extend(
            preds.cpu().numpy()
        )

        targets.extend(
            labels.cpu().numpy()
        )

print("Inferencia terminada")

100%|█████████████████████████████████████████████| 2/2 [00:01<00:00,  1.63it/s]

Inferencia terminada


In [65]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

print("=" * 60)
print("VISUALBERT + SAMPLING (6 frames)")
print("=" * 60)

print(f"Accuracy: {accuracy_score(targets, predictions):.4f}")

print("\nClassification Report:\n")

print(
    classification_report(
        targets,
        predictions,
        target_names=[
            "basketball",
            "soccer",
            "swimming",
            "tennis"
        ],
        digits=4
    )
)

print("\nConfusion Matrix:\n")

print(
    confusion_matrix(
        targets,
        predictions
    )
)

VISUALBERT + SAMPLING (6 frames)
Accuracy: 0.1667

Classification Report:

              precision    recall  f1-score   support

  basketball     0.0000    0.0000    0.0000         5
      soccer     0.0000    0.0000    0.0000         5
    swimming     0.1667    1.0000    0.2857         3
      tennis     0.0000    0.0000    0.0000         5

    accuracy                         0.1667        18
   macro avg     0.0417    0.2500    0.0714        18
weighted avg     0.0278    0.1667    0.0476        18


Confusion Matrix:

[[0 0 5 0]
 [0 0 5 0]
 [0 0 3 0]
 [0 0 5 0]]


/usr/local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


#INTERPRETACION
¿Es válido este resultado?
Cuando se reduce el contexto visual de 12 a 6 frames sin reentrenamiento, VisualBERT pierde gran parte de su capacidad de clasificación, mientras que LXMERT mantiene su rendimiento.

| Modelo | Original | Shuffle | Sampling-6 | Observación |
|---------|---------:|---------:|-----------:|-------------|
| CLIP (Zero-shot) | — | — | — | Modelo base sin ajuste fino. |
| VisualBERT | **0.7778** | **0.7778** | **0.1667** | La reducción de 12 a 6 frames disminuyó significativamente el rendimiento. |
| LXMERT | **0.8889** | **0.8889** | **0.8889** | Mantuvo el mismo rendimiento frente a ambas perturbaciones. |

# Conclusión

En este cuaderno se desarrolló un pipeline multimodal para la clasificación de deportes utilizando un subconjunto de **MSR-VTT**, donde cada video fue representado mediante una secuencia de **12 frames**. El experimento incluyó la extracción de características visuales con **CLIP**, la generación automática de descripciones mediante **BLIP**, la construcción del conjunto de datos multimodal y el entrenamiento de una **capa de clasificación** sobre los modelos preentrenados **VisualBERT** y **LXMERT**.

Inicialmente, se evaluó el uso de **CLIP** para seleccionar y relacionar la información visual con las descripciones disponibles. Debido a que el desempeño obtenido fue limitado Mejor similitud promedio por frame-caption: 0.346908289525244. Ranking guardado:/workspace/Exposicion3/notebook_MSR_VTT/outputs/tables/sports33_test_context_12_clip_ranking.csv
Filas: 2880. Por eso, se incorporó **BLIP** para generar descripciones basadas en el contenido de los **12 frames** de cada muestra, obteniendo representaciones textuales más acordes con la información visual utilizada durante el entrenamiento.

Los resultados permitieron comparar el desempeño de ambas arquitecturas utilizando métricas de clasificación y perturbaciones temporales. En el conjunto de prueba, **LXMERT** obtuvo una mayor precisión que **VisualBERT**. Asimismo, LXMERT mantuvo su rendimiento frente a las perturbaciones **Shuffle** y **Sampling-6**, mientras que VisualBERT mostró una disminución importante de su precisión al reducir el número de frames disponibles.

En conjunto, este cuaderno presenta una metodología reproducible para la construcción y evaluación de modelos multimodales utilizando **secuencias de frames representativas de videos**, permitiendo analizar la robustez de diferentes arquitecturas frente a perturbaciones en la información temporal.